# H&M Socks — RAG Pipeline (Optimized)

This notebook extends the existing H&M socks RAG pipeline with an **experimental
optimization of two components only**:

1. **Embedding model** — currently `BAAI/bge-small-en-v1.5`
2. **Generation model** — currently `Qwen/Qwen2.5-1.5B-Instruct`

Everything else is preserved exactly: the product-level document strategy
(`article_id` → `product_code` → product document), Qdrant indexing, metadata
filtering, variant resolution, and Cross-Encoder reranking.

## Notebook structure

| Part | Contents |
|---|---|
| **Part 1** | The existing pipeline, unchanged. Defines all data, functions, and the baseline models. |
| **Part 2** | Embedding model experiment — baseline vs. candidates, measured on a fixed evaluation set. |
| **Part 3** | Generation model experiment — baseline vs. candidates on identical retrieved context. |
| **Part 4** | Final pipeline using the experimentally selected models, plus before/after comparison. |

## ⚠️ Important note on results

**The experiment cells in Parts 2–4 have not been executed.** They were written
in an environment with no GPU, no Hugging Face access, and no copy of the H&M
dataset, so no numbers could be produced. Every metric cell is left with **empty
output on purpose** rather than filled with invented values.

The model selection in Part 4 is **computed at runtime from the measured
metrics** — it is not hardcoded. Run Parts 1–3 on Kaggle (GPU + internet
enabled) and the notebook will print its own decision and summary based on what
it actually measures on your data.


---

# Part 1 — Existing Pipeline (Baseline)

The cells below are the original notebook, unchanged. They build the product
documents, index them in Qdrant with the baseline embedding model, and define
`retrieve_products()`, `rerank_products()`, `build_context()`, `build_prompt()`,
and `generate_response()`.

Part 2 onward reuses these functions rather than reimplementing them.


# H&M Socks — RAG Pipeline

**Project:** RAG-based retrieval system for H&M socks products.

This notebook implements the Retrieval-Augmented Generation (RAG) pipeline for the H&M socks e-commerce dataset: turning the sock catalog into searchable product documents, embedding and indexing them in a vector database, and retrieving matching products (down to the exact variant) for a natural-language query.

It is the third notebook in the project, and builds directly on the cleaned dataset produced by `02_Data_Validation_Cleaning_HM_Socks.ipynb` (which itself follows `01_EDA_HM_Socks.ipynb`). All retrieval, embedding, indexing, and variant-resolution logic below is unchanged from the original implementation.

## 2. RAG Data Preparation

This notebook now loads the **cleaned, validated** sock dataset produced by `02_Data_Validation_Cleaning_HM_Socks.ipynb`, instead of re-deriving and re-cleaning it from the raw H&M tables. That notebook already handled missing values, NaNs, and duplicates — this notebook just consumes the result.

Everything downstream (product document creation, embedding, indexing, retrieval, variant resolution) is otherwise unchanged.

In [1]:
import pandas as pd

CLEAN_DATA_PATH = "/kaggle/input/datasets/ehabashraf/hm-stokc-clean/hm_socks_clean.csv"

socks = pd.read_csv(CLEAN_DATA_PATH)

print("Loaded cleaned sock dataset:", socks.shape)
socks.head()

Loaded cleaned sock dataset: (1889, 26)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,inferred_gender
0,111565003,111565,20 den 1p Stockings,302,Socks,Socks & Tights,1010016,Solid,13,Beige,...,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo...",Women
1,148033006,148033,Nouvelle 1p Stay Up,302,Socks,Socks & Tights,1010016,Solid,13,Beige,...,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Semi shiny stay-ups with a wide lace trim at t...,Women
2,156224001,156224,Box 4p Socks,302,Socks,Socks & Tights,1010016,Solid,13,Beige,...,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Semi-matte socks with a short shaft. 20 denier.,Women
3,160442007,160442,3p Sneaker Socks,302,Socks,Socks & Tights,1010016,Solid,9,Black,...,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Short, fine-knit socks designed to be hidden b...",Women
4,160442010,160442,3p Sneaker Socks,302,Socks,Socks & Tights,1010016,Solid,10,White,...,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Short, fine-knit socks designed to be hidden b...",Women


## 3. Product and Variant Modeling

Two identifiers matter for this pipeline, and they operate at different levels:

- **`article_id`** — a *variant*-level identifier: one specific color/size SKU.
- **`product_code`** — a *product/style*-level identifier: the group of variants that share the same design.

The RAG pipeline retrieves at the **`product_code`** level, because that's the level at which a semantically meaningful product description exists (name, type, description, the set of available colors, etc.). Once a matching product is found, `article_id` is used only at the very end to resolve the exact variant (e.g. a specific color) the user asked for.

This is exactly the transition observed during EDA:

**1,889 sock variants → 474 unique products**

## 4. Product Document Creation

Each unique `product_code` is turned into a single **product document**: a free-text description (for embedding) plus a structured metadata dict (for filtering). The fields included are:

- Product Name
- Product Type
- Product Group
- Gender (inferred, see Section 2)
- Department
- Section
- Garment Group
- Colors
- Perceived Colors
- Description

These fields are chosen because they carry the semantic information a shopper would actually search on (e.g. "black men's socks", "soft cotton socks"), while the structured metadata version of the same fields is kept alongside so it can later be used for exact-match filtering (gender, color, product type, ...) on top of semantic search.

In [2]:
def build_product_document(group):
    first = group.iloc[0]

    colors = sorted(
        group["colour_group_name"]
        .dropna()
        .unique()
        .tolist()
    )

    perceived_colors = sorted(
        group["perceived_colour_master_name"]
        .dropna()
        .unique()
        .tolist()
    )

    article_ids = (
        group["article_id"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    description = (
        group["detail_desc"]
        .dropna()
        .astype(str)
        .loc[lambda x: x.str.strip() != ""]
        .iloc[0]
        if group["detail_desc"].notna().any()
        else ""
    )

    gender_values = group["inferred_gender"].dropna().unique().tolist()

    if len(gender_values) == 1:
        gender = gender_values[0]
    else:
        gender = "Unisex/Unknown"

    document = f"""
Product Name: {first['prod_name']}
Product Type: {first['product_type_name']}
Product Group: {first['product_group_name']}
Gender: {gender}
Department: {first['department_name']}
Section: {first['section_name']}
Garment Group: {first['garment_group_name']}

Colors: {', '.join(colors)}
Perceived Colors: {', '.join(perceived_colors)}

Description:
{description}
""".strip()

    metadata = {
        "product_code": int(first["product_code"]),
        "article_ids": article_ids,
        "product_name": first["prod_name"],
        "product_type": first["product_type_name"],
        "product_group": first["product_group_name"],
        "gender": gender,
        "department": first["department_name"],
        "section": first["section_name"],
        "garment_group": first["garment_group_name"],
        "colors": colors,
        "perceived_colors": perceived_colors,
    }

    return {
        "document": document,
        "metadata": metadata
    }

In [3]:
product_code = 783707

group = socks[
    socks["product_code"] == product_code
]

document, metadata = build_product_document(group)

print("DOCUMENT")
print("=" * 80)
print(document)

print("\nMETADATA")
print("=" * 80)
print(metadata)

DOCUMENT
document

METADATA
metadata


In [4]:
product_documents = []

for product_code, group in socks.groupby("product_code"):
    product_documents.append(
        build_product_document(group)
    )

print("Number of product documents:", len(product_documents))

Number of product documents: 474


In [5]:
for item in product_documents:
    if item["metadata"]["product_code"] == 783707:
        print(item["document"])
        print("\nMETADATA:")
        print(item["metadata"])
        break

Product Name: 1pk Fun
Product Type: Socks
Product Group: Socks & Tights
Gender: Men
Department: Socks Wall
Section: Men Underwear
Garment Group: Socks and Tights

Colors: Beige, Black, Blue, Dark Blue, Dark Green, Dark Grey, Dark Orange, Dark Purple, Dark Turquoise, Greenish Khaki, Grey, Light Blue, Light Green, Light Grey, Light Purple, Light Turquoise, Light Yellow, Orange, Pink, Purple, Red, Turquoise, White
Perceived Colors: Beige, Black, Blue, Green, Grey, Khaki green, Lilac Purple, Orange, Pink, Red, Turquoise, Unknown, White, Yellow

Description:
Socks in a soft, jacquard-knit cotton blend with elasticated tops.

METADATA:
{'product_code': 783707, 'article_ids': ['783707002', '783707004', '783707005', '783707012', '783707015', '783707019', '783707021', '783707026', '783707028', '783707029', '783707030', '783707031', '783707032', '783707033', '783707034', '783707035', '783707036', '783707037', '783707038', '783707039', '783707041', '783707043', '783707045', '783707046', '78370704

In [6]:
variant_columns = [
    "article_id",
    "product_code",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "perceived_colour_master_name",
    "department_name",
    "section_name",
    "garment_group_name",
    "detail_desc"
]

sock_variants = socks[variant_columns].copy()

print(sock_variants.shape)
display(sock_variants.head(10))

(1889, 11)


,article_id,product_code,prod_name,product_type_name,product_group_name,colour_group_name,perceived_colour_master_name,department_name,section_name,garment_group_name,detail_desc
0,111565003,111565,20 den 1p Stockings,Socks,Socks & Tights,Beige,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
1,148033006,148033,Nouvelle 1p Stay Up,Socks,Socks & Tights,Beige,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi shiny stay-ups with a wide lace trim at t...
2,156224001,156224,Box 4p Socks,Socks,Socks & Tights,Beige,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi-matte socks with a short shaft. 20 denier.
3,160442007,160442,3p Sneaker Socks,Socks,Socks & Tights,Black,Black,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
4,160442010,160442,3p Sneaker Socks,Socks,Socks & Tights,White,White,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
5,160442042,160442,Sneaker 3p Socks,Socks,Socks & Tights,Light Grey,Grey,Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
6,160442043,160442,3p Sneaker Socks,Socks,Socks & Tights,Light Grey,Grey,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
7,162074062,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Pink,Pink,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
8,162074069,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Grey,Grey,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
9,162074071,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Black,Grey,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.


In [7]:
print(product_documents[0]["document"])
print()
print(product_documents[0]["metadata"])

Product Name: 20 den 1p Stockings
Product Type: Socks
Product Group: Socks & Tights
Gender: Women
Department: Tights basic
Section: Womens Nightwear, Socks & Tigh
Garment Group: Socks and Tights

Colors: Beige
Perceived Colors: Beige

Description:
Semi shiny nylon stockings with a wide, reinforced trim at the top. Use with a suspender belt. 20 denier.

{'product_code': 111565, 'article_ids': ['111565003'], 'product_name': '20 den 1p Stockings', 'product_type': 'Socks', 'product_group': 'Socks & Tights', 'gender': 'Women', 'department': 'Tights basic', 'section': 'Womens Nightwear, Socks & Tigh', 'garment_group': 'Socks and Tights', 'colors': ['Beige'], 'perceived_colors': ['Beige']}


**Validation:** a quick check that all product documents were built and that none of the required metadata fields are missing.

In [8]:
print("Products:", len(product_documents))

Products: 474


In [9]:
from collections import Counter

gender_counts = Counter(
    item["metadata"]["gender"]
    for item in product_documents
)

print(gender_counts)

Counter({'Unisex/Unknown': 234, 'Women': 151, 'Men': 89})


In [10]:
required_fields = [
    "product_code",
    "product_name",
    "product_type",
    "product_group",
    "gender",
    "department",
    "section",
    "garment_group",
    "colors",
    "perceived_colors"
]

for field in required_fields:
    missing = sum(
        1 for item in product_documents
        if not item["metadata"].get(field)
    )
    print(f"{field}: {missing} missing")

product_code: 0 missing
product_name: 0 missing
product_type: 0 missing
product_group: 0 missing
gender: 0 missing
department: 0 missing
section: 0 missing
garment_group: 0 missing
colors: 0 missing
perceived_colors: 0 missing


## 5. Embedding

Product documents are text — to make them searchable by meaning rather than by exact keyword match, each document is turned into a dense vector using a sentence-embedding model. This is what enables semantic search later: queries and documents are compared in the same embedding space.

**Embedding model:** `BAAI/bge-small-en-v1.5`
**Embedding dimension:** 384
**Number of product documents embedded:** 474 (`474 × 384` embedding matrix)

In [11]:
!pip install -q sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
documents = [
    item["document"]
    for item in product_documents
]

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding shape: (474, 384)


**Sanity check (in-memory):** before setting up a vector database, the notebook validates the embeddings with a simple in-memory cosine-similarity search over the raw embedding matrix.

In [14]:
from sentence_transformers import util
import torch

def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True,
        convert_to_tensor=True
    )

    # document_embeddings = torch.tensor(embeddings)
    document_embeddings = torch.tensor(embeddings).to(query_embedding.device)

    scores = util.cos_sim(
        query_embedding,
        document_embeddings
    )[0]

    top_results = torch.topk(scores, k=top_k)

    results = []

    for score, idx in zip(
        top_results.values,
        top_results.indices
    ):
        idx = idx.item()

        results.append({
            "score": float(score),
            "product_code": product_documents[idx]["metadata"]["product_code"],
            "product_name": product_documents[idx]["metadata"]["product_name"],
            "document": product_documents[idx]["document"]
        })

    return results

In [15]:
results = semantic_search(
    "soft comfortable cotton socks",
    top_k=5
)

for result in results:
    print("=" * 80)
    print("Score:", result["score"])
    print("Product:", result["product_name"])
    print("Product Code:", result["product_code"])

Score: 0.7060432434082031
Product: COTTON terry socks
Product Code: 867846
Score: 0.7024343609809875
Product: Terry No cuff
Product Code: 753216
Score: 0.6993615627288818
Product: Terry socks 3p
Product Code: 587932
Score: 0.6987034678459167
Product: Heavy Cotton Jacquard 2p socks
Product Code: 666743
Score: 0.697798490524292
Product: Athletic sock
Product Code: 813283


In [16]:
test_queries = [
    "soft comfortable cotton socks",
    "black men's socks",
    "short socks for sneakers",
    "colorful socks",
    "fine knit socks",
    "warm everyday socks"
]

for query in test_queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)

    results = semantic_search(query, top_k=3)

    for r in results:
        print(
            f"{r['score']:.4f} | "
            f"{r['product_name']} | "
            f"{r['product_code']}"
        )


QUERY: soft comfortable cotton socks
0.7060 | COTTON terry socks | 867846
0.7024 | Terry No cuff | 753216
0.6994 | Terry socks 3p | 587932

QUERY: black men's socks
0.7607 | Lawrence Shaftless Sock 5-p | 861850
0.7572 | 20 pk regular basic | 791428
0.7553 | Bernard Crew Sock 5-p | 861851

QUERY: short socks for sneakers
0.7187 | 7pk basic R sneaker socks | 575347
0.7175 | Everyday Sneaker 5p | 624749
0.7137 | 3p Sneaker Socks | 160442

QUERY: colorful socks
0.7619 | Singel ankle sock 1-p | 863248
0.7613 | 1 pk Fun | 667411
0.7577 | Fun Chenille 2p Lounge sock | 408649

QUERY: fine knit socks
0.7249 | 5pk Basic Rib | 847570
0.7214 | Lars-Inge 2-pack | 639737
0.7203 | Wool 2p Kneehigh | 715424

QUERY: warm everyday socks
0.6906 | Everyday Sneaker 5p | 624749
0.6874 | Everyday 5p shaftless | 373493
0.6856 | I LOVE sock | 511213


The original notebook re-loads the embedding model and recomputes the embeddings a second time at this point in the workflow. That cell is preserved below unchanged, even though it duplicates the computation above.

In [17]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

documents = [
    item["document"]
    for item in product_documents
]

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding shape: (474, 384)


## 6. Vector Database / Indexing

### Embedding & Indexing

[Qdrant](https://qdrant.tech/) is used as the vector database. For each product:

- The **vector** stored is the product document's embedding (384-dim).
- The **payload** (metadata) stored alongside it is the structured metadata dict from Section 4 — product name/type/group, gender, department, section, garment group, colors, perceived colors, and the list of `article_id`s for that product — which is what enables filtered/hybrid retrieval later.

All **474** product vectors are indexed into the collection.

In [18]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 23.5 MB/s eta 0:00:00


In [19]:
# qdrant_client.delete_collection(
# #     collection_name=collection_name
# # )

In [20]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(":memory:")

print("Qdrant client ready")

Qdrant client ready


In [21]:
from qdrant_client.models import VectorParams, Distance

collection_name = "hm_socks_products"

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print(qdrant_client.get_collection(collection_name))

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, pr

In [22]:
from qdrant_client.models import PointStruct

points = []

for idx, item in enumerate(product_documents):

    metadata = item["metadata"]

    payload = {
        "product_code": metadata["product_code"],
        "article_ids": metadata["article_ids"],
        "product_name": metadata["product_name"],
        "product_type": metadata["product_type"],
        "product_group": metadata["product_group"],
        "gender": metadata["gender"],
        "department": metadata["department"],
        "section": metadata["section"],
        "garment_group": metadata["garment_group"],
        "colors": metadata["colors"],
        "perceived_colors": metadata["perceived_colors"],
    }

    points.append(
        PointStruct(
            id=idx,
            vector=embeddings[idx].tolist(),
            payload=payload
        )
    )

print("Points:", len(points))

Points: 474


In [23]:
qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

print("Uploaded successfully")

Uploaded successfully


In [24]:
print(
    qdrant_client.count(
        collection_name=collection_name,
        exact=True
    )
)

count=474


## 7. Retrieval

This is the semantic retrieval stage of the RAG pipeline:

```
User Query
   ↓
Query Embedding
   ↓
Qdrant Vector Search
   ↓
Semantic Candidate Retrieval
```

A query string is embedded with the same model used for the product documents, then compared against the indexed vectors in Qdrant to return the closest matching products.

In [25]:
def qdrant_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        limit=top_k
    )

    return results.points

In [26]:
results = qdrant_search(
    "black men's socks",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['gender']} | "
        f"{result.payload['colors']}"
    )

0.7607 | Lawrence Shaftless Sock 5-p | Men | ['Black', 'Dark Grey', 'Grey', 'White']
0.7572 | 20 pk regular basic | Men | ['Black']
0.7553 | Bernard Crew Sock 5-p | Men | ['Black', 'Dark Grey', 'White']
0.7537 | 5pk regular Placement1 | Men | ['Black']
0.7531 | Lawrence reflective | Men | ['Black', 'White']


## 8. Metadata Filtering / Hybrid Retrieval

**Semantic Search** (Section 7 above) ranks products purely by embedding similarity to the query. **Structured Metadata Filtering** narrows the candidate set to only products that match exact structured criteria — here:

- Gender
- Color
- Product Type
- Department
- Section

Combining the two — semantic ranking constrained by structured filters — is **hybrid retrieval**, implemented below.

In [27]:
from qdrant_client.models import Filter, FieldCondition, MatchValue


def qdrant_search_filtered(
    query,
    gender=None,
    color=None,
    top_k=5
):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    conditions = []

    if gender:
        conditions.append(
            FieldCondition(
                key="gender",
                match=MatchValue(value=gender)
            )
        )

    if color:
        conditions.append(
            FieldCondition(
                key="colors",
                match=MatchValue(value=color)
            )
        )

    query_filter = None

    if conditions:
        query_filter = Filter(
            must=conditions
        )

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        query_filter=query_filter,
        limit=top_k
    )

    return results.points

In [28]:
results = qdrant_search_filtered(
    "black men's socks",
    gender="Men",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['gender']} | "
        f"{result.payload['colors']}"
    )

0.7607 | Lawrence Shaftless Sock 5-p | Men | ['Black', 'Dark Grey', 'Grey', 'White']
0.7572 | 20 pk regular basic | Men | ['Black']
0.7553 | Bernard Crew Sock 5-p | Men | ['Black', 'Dark Grey', 'White']
0.7537 | 5pk regular Placement1 | Men | ['Black']
0.7531 | Lawrence reflective | Men | ['Black', 'White']


In [29]:
results = qdrant_search_filtered(
    "men's socks",
    gender="Men",
    color="Black",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['gender']} | "
        f"{result.payload['colors']}"
    )

0.7697 | Lawrence Shaftless Sock 5-p | Men | ['Black', 'Dark Grey', 'Grey', 'White']
0.7680 | 20 pk regular basic | Men | ['Black']
0.7655 | Roy sock(1) | Men | ['Black', 'White']
0.7652 | Roy socks 5-pack | Men | ['Black', 'White']
0.7647 | Lawrence 7pack. | Men | ['Black']


In [30]:
from qdrant_client.models import Filter, FieldCondition, MatchValue


def qdrant_hybrid_search(
    query,
    gender=None,
    color=None,
    product_type=None,
    department=None,
    section=None,
    top_k=5
):
    # 1. Encode semantic query
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    # 2. Build structured filters
    conditions = []

    if gender:
        conditions.append(
            FieldCondition(
                key="gender",
                match=MatchValue(value=gender)
            )
        )

    if color:
        conditions.append(
            FieldCondition(
                key="colors",
                match=MatchValue(value=color)
            )
        )

    if product_type:
        conditions.append(
            FieldCondition(
                key="product_type",
                match=MatchValue(value=product_type)
            )
        )

    if department:
        conditions.append(
            FieldCondition(
                key="department",
                match=MatchValue(value=department)
            )
        )

    if section:
        conditions.append(
            FieldCondition(
                key="section",
                match=MatchValue(value=section)
            )
        )

    query_filter = None

    if conditions:
        query_filter = Filter(must=conditions)

    # 3. Search Qdrant
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        query_filter=query_filter,
        limit=top_k
    )

    return results.points

In [31]:
results = qdrant_hybrid_search(
    query="short socks for sneakers",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['gender']} | "
        f"{result.payload['colors']}"
    )

0.7187 | 7pk basic R sneaker socks | Men | ['Black', 'Dark Blue', 'Light Turquoise', 'White']
0.7175 | Everyday Sneaker 5p | Women | ['Black']
0.7137 | 3p Sneaker Socks | Women | ['Black', 'Light Grey', 'White']
0.7097 | Short 5p Socks(1) | Women | ['White']
0.7095 | 10pk sneaker sock basic | Men | ['White']


In [32]:
results = qdrant_hybrid_search(
    query="short socks for sneakers",
    gender="Men",
    color="Black",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['gender']} | "
        f"{result.payload['colors']}"
    )

0.7187 | 7pk basic R sneaker socks | Men | ['Black', 'Dark Blue', 'Light Turquoise', 'White']
0.7010 | 1 pk Sneaker | Men | ['Black', 'Dark Blue', 'Dark Turquoise', 'Grey', 'Turquoise']
0.6937 | Lawrence Reflective Low Sock 3 | Men | ['Black', 'White']
0.6836 | Raz shaftless sock 5-pack. | Men | ['Black', 'Grey', 'White']
0.6834 | Roy socks 5-pack | Men | ['Black', 'White']


In [33]:
results = qdrant_hybrid_search(
    query="comfortable everyday socks",
    gender="Men",
    product_type="Socks",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['product_type']} | "
        f"{result.payload['gender']}"
    )

0.6624 | 1pk Smart | Socks | Men
0.6610 | Lawrence Reflective Low Sock 3 | Socks | Men
0.6600 | Lawrence Shaftless Sock 5-p | Socks | Men
0.6593 | Lawrence reflective | Socks | Men
0.6592 | 5pk Sport Sock Solid | Socks | Men


In [34]:
def validate_filters(results, gender=None, color=None, product_type=None):
    checks = []

    for result in results:
        payload = result.payload

        if gender is not None:
            checks.append(
                payload["gender"] == gender
            )

        if color is not None:
            checks.append(
                color in payload["colors"]
            )

        if product_type is not None:
            checks.append(
                payload["product_type"] == product_type
            )

    return all(checks) if checks else True

In [35]:
results = qdrant_hybrid_search(
    query="short socks for sneakers",
    gender="Men",
    color="Black",
    top_k=10
)

print(
    "Filters valid:",
    validate_filters(
        results,
        gender="Men",
        color="Black"
    )
)

Filters valid: True


## Time To Work

## 9. Variant Resolution

Product-level retrieval returns a `product_code`, but a shopper ultimately needs an exact, purchasable variant:

```
Product-level retrieval
        ↓
   product_code
        ↓
   Variant lookup
        ↓
  Requested color
        ↓
   Exact article_id
```

Product retrieval and variant resolution are kept as **separate concerns**: retrieval finds the right *product*, while variant resolution finds the right *SKU* for that product given a requested attribute (e.g. color). This keeps the embedding/index focused on product-level semantics, while exact variant lookup stays a simple structured lookup over the article-level table.

Below, the retrieval and filtering logic built above is packaged into a reusable `ProductRetriever` class that combines semantic search, metadata filtering, and variant resolution. Its `variant_lookup` table (built first) is derived from the `sock_variants` table created in Section 4.

In [36]:
variant_lookup = (
    sock_variants[
        [
            "article_id",
            "product_code",
            "prod_name",
            "colour_group_name",
            "perceived_colour_master_name"
        ]
    ]
    .copy()
)

variant_lookup["article_id"] = (
    variant_lookup["article_id"]
    .astype(str)
)

variant_lookup["product_code"] = (
    variant_lookup["product_code"]
    .astype(int)
)

In [37]:
from qdrant_client.models import Filter, FieldCondition, MatchValue


class ProductRetriever:

    def __init__(
        self,
        client,
        collection_name,
        embedding_model,
        variant_lookup
    ):
        self.client = client
        self.collection_name = collection_name
        self.embedding_model = embedding_model
        self.variant_lookup = variant_lookup

    def search(
        self,
        query,
        gender=None,
        color=None,
        product_type=None,
        department=None,
        section=None,
        top_k=5
    ):
        # 1. Encode semantic query
        query_embedding = self.embedding_model.encode(
            query,
            normalize_embeddings=True
        )

        # 2. Build metadata filters
        conditions = []

        if gender:
            conditions.append(
                FieldCondition(
                    key="gender",
                    match=MatchValue(value=gender)
                )
            )

        if color:
            conditions.append(
                FieldCondition(
                    key="colors",
                    match=MatchValue(value=color)
                )
            )

        if product_type:
            conditions.append(
                FieldCondition(
                    key="product_type",
                    match=MatchValue(value=product_type)
                )
            )

        if department:
            conditions.append(
                FieldCondition(
                    key="department",
                    match=MatchValue(value=department)
                )
            )

        if section:
            conditions.append(
                FieldCondition(
                    key="section",
                    match=MatchValue(value=section)
                )
            )

        # 3. Create Qdrant filter
        query_filter = None

        if conditions:
            query_filter = Filter(must=conditions)

        # 4. Search Qdrant
        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding.tolist(),
            query_filter=query_filter,
            limit=top_k
        )

        return results.points

    def resolve_variants(
        self,
        product_code,
        color=None
    ):
        # Get all variants for this product
        variants = self.variant_lookup[
            self.variant_lookup["product_code"] == product_code
        ].copy()

        # If user requested a color,
        # keep only exact color matches
        if color:
            variants = variants[
                variants["colour_group_name"].str.lower()
                == color.lower()
            ]

        return variants

In [38]:
retriever = ProductRetriever(
    client=qdrant_client,
    collection_name=collection_name,
    embedding_model=embedding_model,
    variant_lookup=variant_lookup
)

In [39]:
results = retriever.search(
    query="black men's socks for sneakers",
    gender="Men",
    color="Black",
    top_k=5
)

for result in results:
    print(
        f"{result.score:.4f} | "
        f"{result.payload['product_name']} | "
        f"{result.payload['product_code']} | "
        f"{result.payload['article_ids'][:5]} | "
        f"{result.payload['colors']}"
    )

0.7548 | 1 pk Sneaker | 720577 | ['720577008', '720577020', '720577022', '720577023', '720577025'] | ['Black', 'Dark Blue', 'Dark Turquoise', 'Grey', 'Turquoise']
0.7496 | 7pk basic R sneaker socks | 575347 | ['575347003', '575347005', '575347009', '575347010', '575347011'] | ['Black', 'Dark Blue', 'Light Turquoise', 'White']
0.7384 | Lawrence Shaftless Sock 5-p | 861850 | ['861850001', '861850002', '861850003', '861850004', '861850006'] | ['Black', 'Dark Grey', 'Grey', 'White']
0.7337 | Lawrence reflective | 617787 | ['617787001', '617787002'] | ['Black', 'White']
0.7334 | Lawrence training sock | 365439 | ['365439048', '365439051', '365439052', '365439057', '365439058'] | ['Black', 'Dark Blue', 'Dark Green', 'Greenish Khaki', 'Grey']


In [40]:
info = qdrant_client.get_collection(collection_name)

print("Points:", info.points_count)

Points: 474


In [41]:
test_point = qdrant_client.retrieve(
    collection_name=collection_name,
    ids=[0]
)[0]

print(test_point.payload)

{'product_code': 111565, 'article_ids': ['111565003'], 'product_name': '20 den 1p Stockings', 'product_type': 'Socks', 'product_group': 'Socks & Tights', 'gender': 'Women', 'department': 'Tights basic', 'section': 'Womens Nightwear, Socks & Tigh', 'garment_group': 'Socks and Tights', 'colors': ['Beige'], 'perceived_colors': ['Beige']}


The original notebook also has a standalone `resolve_variants` function (equivalent logic to `ProductRetriever.resolve_variants`), tested below alongside the class method.

In [42]:
def resolve_variants(
    product_code,
    color=None
):
    variants = variant_lookup[
        variant_lookup["product_code"] == product_code
    ].copy()

    if color:
        variants = variants[
            variants["colour_group_name"].str.lower()
            == color.lower()
        ]

    return variants

In [43]:
variants = resolve_variants(
    product_code=720577,
    color="Black"
)

variants

,article_id,product_code,prod_name,colour_group_name,perceived_colour_master_name
1279,720577008,720577,1 pk Sneaker,Black,Black


In [44]:
variants = retriever.resolve_variants(
    product_code=720577,
    color="Black"
)

print(variants)

     article_id  product_code     prod_name colour_group_name  \
1279  720577008        720577  1 pk Sneaker             Black   

     perceived_colour_master_name  
1279                        Black  


In [45]:
variants = retriever.resolve_variants(
    product_code=720577
)

print(variants)

     article_id  product_code     prod_name colour_group_name  \
1279  720577008        720577  1 pk Sneaker             Black   
1280  720577020        720577  1 pk Sneaker    Dark Turquoise   
1281  720577022        720577  1 pk Sneaker         Dark Blue   
1282  720577023        720577  1 pk Sneaker              Grey   
1283  720577025        720577  1 pk Sneaker         Turquoise   

     perceived_colour_master_name  
1279                        Black  
1280                    Turquoise  
1281                         Blue  
1282                         Grey  
1283                    Turquoise  


## 10. End-to-End Retrieval

`retrieve_products()` combines every stage above into a single call:

- Qdrant retrieval (semantic search, over-fetching `top_k * 3` candidates)
- Metadata filtering (gender/color/product type/department/section)
- Variant resolution (`article_id` lookup for the requested color)
- Candidate expansion/trimming down to the final `top_k` products, each with its resolved variants attached

Tested against the test cases already present in the notebook: Men black sneaker socks, Women white socks, Colorful socks, Men everyday socks, and Blue socks.

In [46]:
def retrieve_products(
    query,
    gender=None,
    color=None,
    product_type=None,
    department=None,
    section=None,
    top_k=5
):
    # Retrieve more candidates than needed
    candidate_k = top_k * 3

    results = retriever.search(
        query=query,
        gender=gender,
        color=color,
        product_type=product_type,
        department=department,
        section=section,
        top_k=candidate_k
    )

    final_results = []

    for result in results:

        payload = result.payload
        product_code = payload["product_code"]

        variants = retriever.resolve_variants(
            product_code=product_code,
            color=color
        )

        # Skip if requested color doesn't exist
        if color and variants.empty:
            continue

        final_results.append({
            "score": float(result.score),
            "product_code": product_code,
            "product_name": payload["product_name"],
            "product_type": payload["product_type"],
            "product_group": payload["product_group"],
            "gender": payload["gender"],
            "department": payload["department"],
            "section": payload["section"],
            "garment_group": payload["garment_group"],
            "colors": payload["colors"],
            "perceived_colors": payload["perceived_colors"],
            "variants": variants[
                [
                    "article_id",
                    "colour_group_name",
                    "perceived_colour_master_name"
                ]
            ].to_dict("records")
        })

        if len(final_results) >= top_k:
            break

    return final_results

In [47]:
results = retrieve_products(
    query="short socks for sneakers",
    gender="Men",
    color="Black",
    top_k=5
)

for item in results:
    print(
        item["product_name"],
        "|",
        item["product_code"],
        "|",
        item["variants"]
    )

7pk basic R sneaker socks | 575347 | [{'article_id': '575347003', 'colour_group_name': 'Black', 'perceived_colour_master_name': 'Black'}]
1 pk Sneaker | 720577 | [{'article_id': '720577008', 'colour_group_name': 'Black', 'perceived_colour_master_name': 'Black'}]
Lawrence Reflective Low Sock 3 | 862122 | [{'article_id': '862122001', 'colour_group_name': 'Black', 'perceived_colour_master_name': 'Black'}]
Raz shaftless sock 5-pack. | 892193 | [{'article_id': '892193001', 'colour_group_name': 'Black', 'perceived_colour_master_name': 'Black'}]
Roy socks 5-pack | 890347 | [{'article_id': '890347002', 'colour_group_name': 'Black', 'perceived_colour_master_name': 'Black'}]


In [48]:
test_cases = [
    {
        "name": "Men black sneaker socks",
        "query": "short socks for sneakers",
        "gender": "Men",
        "color": "Black"
    },
    {
        "name": "Women white socks",
        "query": "comfortable white socks",
        "gender": "Women",
        "color": "White"
    },
    {
        "name": "Colorful socks",
        "query": "colorful fun socks",
        "gender": None,
        "color": None
    },
    {
        "name": "Men everyday socks",
        "query": "comfortable everyday socks",
        "gender": "Men",
        "color": None
    },
    {
        "name": "Blue socks",
        "query": "blue socks",
        "gender": None,
        "color": "Blue"
    }
]

In [49]:
for test in test_cases:

    print("=" * 80)
    print(test["name"])

    results = retrieve_products(
        query=test["query"],
        gender=test["gender"],
        color=test["color"],
        top_k=5
    )

    for i, item in enumerate(results, 1):

        print(
            f"{i}. "
            f"{item['product_name']} | "
            f"{item['product_code']} | "
            f"score={item['score']:.4f}"
        )

        for variant in item["variants"]:
            print(
                f"   → {variant['article_id']} | "
                f"{variant['colour_group_name']}"
            )

Men black sneaker socks
1. 7pk basic R sneaker socks | 575347 | score=0.7187
   → 575347003 | Black
2. 1 pk Sneaker | 720577 | score=0.7010
   → 720577008 | Black
3. Lawrence Reflective Low Sock 3 | 862122 | score=0.6937
   → 862122001 | Black
4. Raz shaftless sock 5-pack. | 892193 | score=0.6836
   → 892193001 | Black
5. Roy socks 5-pack | 890347 | score=0.6834
   → 890347002 | Black
Women white socks
1. 1p Placement Socks | 836809 | score=0.7218
   → 836809003 | White
   → 836809004 | White
   → 836809005 | White
2. 1p Casual Socks | 807239 | score=0.7210
   → 807239005 | White
3. OL Excl 90's Dance 1p Sock | 788102 | score=0.7182
   → 788102001 | White
4. Coconut 2-pack socks | 910177 | score=0.7125
   → 910177001 | White
5. Short 5p Socks(1) | 892079 | score=0.7120
   → 892079001 | White
Colorful socks
1. 1 pk Fun | 667411 | score=0.7453
   → 667411001 | Light Yellow
   → 667411002 | Pink
   → 667411003 | Dark Green
   → 667411005 | Dark Blue
   → 667411008 | Dark Blue
   → 6674110

## RAG Pipeline Stage Summary

| Stage | Description | Status |
|---|---|---|
| 1. Data Ingestion | Raw data → cleaned/prepared data | ✅ Implemented |
| 2. Embedding & Indexing | Documents → embeddings → Qdrant | ✅ Implemented |
| 3. Retrieval | Query → embedding → relevant candidates | ✅ Implemented |
| 4. Reranking | Candidates → Cross-Encoder → better ranking | ❌ Not implemented |
| 5. Augmentation | Retrieved information → context/prompt | ❌ Not implemented |
| 6. Generation | Context + Query → LLM → final answer | ❌ Not implemented |

## 11. Reranking

### Reranking — Next Stage

This stage is **not implemented yet** in the current notebook.

Conceptually, once Qdrant returns a larger candidate set (e.g. the top-N products from Section 7/8), a **Cross-Encoder** could score each `Query + Candidate Product` pair directly and reorder the candidates to produce a smaller, more relevant top-K result set. This would sit between the retrieval/filtering stage and the final results returned by `retrieve_products()`, improving ranking quality beyond what embedding similarity alone provides.

## Why Do We Need Reranking?

Vector similarity is useful for retrieving semantically similar products,
but the initial ranking is not always the most precise ranking for an
e-commerce query.

For example, a query such as:

> "short socks for sneakers"

may retrieve several relevant sock products, but some products may be
more specifically relevant to sneakers than others.

The reranker provides a second-stage relevance judgment by evaluating
the query and each candidate product together.

Therefore:

- Qdrant answers: **Which products are likely to be relevant?**
- Reranker answers: **Which of these candidates are the most relevant?**

## Two-Stage Retrieval Architecture

We use a two-stage retrieval architecture:

1. **First Stage — Candidate Retrieval**
   - Qdrant performs semantic vector search.
   - Metadata filters can further restrict the candidates.
   - We retrieve a larger candidate set.

2. **Second Stage — Reranking**
   - A Cross-Encoder evaluates each query-product pair.
   - Candidates are assigned new relevance scores.
   - The candidates are sorted according to the reranker scores.
   - Only the top results are passed to the next RAG stage.

Architecture:

Qdrant
→ Top 15 Candidates
→ Cross-Encoder
→ Top 5 Results

## Loading the Cross-Encoder

We use a pretrained Cross-Encoder model to perform query-product
relevance scoring.

Unlike the embedding model used for vector retrieval, the Cross-Encoder
receives the query and the candidate product together and directly
estimates their relevance.

Model:

`cross-encoder/ms-marco-MiniLM-L-6-v2`

This model is used as a general English relevance model for the current
H&M product dataset.

In [50]:
!pip install -q sentence-transformers

In [51]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

The Cross-Encoder is now loaded and will be used only after the initial
Qdrant retrieval step.

It does not replace the embedding model or Qdrant.

Instead, the two models have different responsibilities:

- **Embedding model:** retrieves candidate products efficiently.
- **Cross-Encoder:** reranks the retrieved candidates more precisely.

## Preparing Product Text for Reranking

The Cross-Encoder needs a textual representation of each candidate
product.

Our Qdrant payload already contains structured product information.

We combine the most relevant product attributes into a single text
representation that can be evaluated together with the user query.

The product representation includes:

- Product Name
- Product Type
- Product Group
- Gender
- Department
- Section
- Garment Group
- Colors
- Perceived Colors

This step does not create new embeddings. The text is only prepared as
input for the Cross-Encoder.

In [52]:
def build_reranker_text(payload):
    return f"""
Product Name: {payload["product_name"]}
Product Type: {payload["product_type"]}
Product Group: {payload["product_group"]}
Gender: {payload["gender"]}
Department: {payload["department"]}
Section: {payload["section"]}
Garment Group: {payload["garment_group"]}
Colors: {", ".join(payload["colors"])}
Perceived Colors: {", ".join(payload["perceived_colors"])}
""".strip()

The `build_reranker_text()` function converts the structured product
metadata into a textual representation.

This allows the Cross-Encoder to evaluate the relationship between:

`User Query + Product Information`

instead of evaluating the product metadata as separate fields.

## Reranking Function

The reranking function receives:

- The original user query
- Retrieved candidate products
- The desired number of final results

For every candidate, we create a query-product pair:

`[query, product_text]`

The Cross-Encoder then assigns a relevance score to each pair.

Finally, candidates are sorted in descending order of their reranking
score and the top results are returned.

In [53]:
def rerank_products(
    query,
    results,
    top_k=5
):
    pairs = []

    for item in results:
        product_text = build_reranker_text(item)
        pairs.append([query, product_text])

    scores = reranker.predict(pairs)

    reranked = []

    for item, score in zip(results, scores):
        item = item.copy()

        item["rerank_score"] = float(score)

        reranked.append(item)

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

The reranking function performs the second stage of our retrieval
pipeline.

The important distinction is:

**Retrieval score**
→ produced by Qdrant during vector search.

**Reranking score**
→ produced by the Cross-Encoder after evaluating the query and product
together.

These scores come from different models and should not be compared
numerically.

## Reranker Evaluation

We will now test the reranker using the existing retrieval pipeline.

Instead of retrieving only the final number of products, we first
retrieve a larger candidate set.

For this experiment:

`Qdrant → Top 15 Candidates`

Then:

`Top 15 → Cross-Encoder → Top 5`

This gives the reranker enough candidates to improve the final ordering.

In [54]:
candidates = retrieve_products(
    query="short socks for sneakers",
    gender="Men",
    color="Black",
    top_k=15
)

Qdrant has retrieved the candidate products using:

- Semantic similarity
- Gender filtering: `Men`
- Color filtering: `Black`

The 15 retrieved products are candidates for the second-stage
reranking process.

They are not yet the final results.

In [55]:
reranked = rerank_products(
    query="short socks for sneakers",
    results=candidates,
    top_k=5
)

The Cross-Encoder has now evaluated the 15 candidate products and
reordered them according to query-product relevance.

Only the top 5 products are retained as the final ranked retrieval
results.

Pipeline:

Qdrant Top 15
→ Cross-Encoder
→ Top 5

In [56]:
for i, item in enumerate(reranked, 1):
    print(
        f"{i}. "
        f"{item['product_name']} | "
        f"product={item['product_code']} | "
        f"retrieval={item['score']:.4f} | "
        f"rerank={item['rerank_score']:.4f}"
    )

1. 7pk basic R sneaker socks | product=575347 | retrieval=0.7187 | rerank=-1.6184
2. Raz shaftless sock 5-pack. | product=892193 | retrieval=0.6836 | rerank=-2.7093
3. Lawrence Shaftless Sock 5-p | product=861850 | retrieval=0.6783 | rerank=-2.7959
4. Lawrence Reflective Low Sock 3 | product=862122 | retrieval=0.6937 | rerank=-2.9119
5. Lawrence training sock | product=365439 | retrieval=0.6728 | rerank=-2.9906


### Interpreting the Results

The output contains two different scores:

- `retrieval`: the original Qdrant semantic similarity score.
- `rerank`: the Cross-Encoder relevance score.

The purpose of the reranker is not to produce a score that is
numerically comparable to the Qdrant score.

Instead, we evaluate whether the ordering of products becomes more
relevant to the user's query.

The final ranking will be used as the retrieved context for the next
stage of the RAG pipeline.

## Testing Reranking Across Multiple Queries

To verify that the reranker works beyond a single example, we evaluate
it using the existing retrieval test cases.

For each query:

1. Retrieve up to 15 candidate products using Qdrant.
2. Apply the existing structured filters.
3. Rerank the candidates using the Cross-Encoder.
4. Keep the top 5 products.

The goal at this stage is to inspect the resulting ranking and verify
that the retrieved products remain relevant to the user's intent.

In [57]:
for test in test_cases:

    print("=" * 80)
    print(test["name"])

    candidates = retrieve_products(
        query=test["query"],
        gender=test["gender"],
        color=test["color"],
        top_k=15
    )

    reranked = rerank_products(
        query=test["query"],
        results=candidates,
        top_k=5
    )

    for i, item in enumerate(reranked, 1):
        print(
            f"{i}. "
            f"{item['product_name']} | "
            f"product={item['product_code']} | "
            f"retrieval={item['score']:.4f} | "
            f"rerank={item['rerank_score']:.4f}"
        )

Men black sneaker socks
1. 7pk basic R sneaker socks | product=575347 | retrieval=0.7187 | rerank=-1.6184
2. Raz shaftless sock 5-pack. | product=892193 | retrieval=0.6836 | rerank=-2.7093
3. Lawrence Shaftless Sock 5-p | product=861850 | retrieval=0.6783 | rerank=-2.7959
4. Lawrence Reflective Low Sock 3 | product=862122 | retrieval=0.6937 | rerank=-2.9119
5. Lawrence training sock | product=365439 | retrieval=0.6728 | rerank=-2.9906
Women white socks
1. 1p Casual Socks | product=807239 | retrieval=0.7210 | rerank=1.2586
2. Merino 2p socks. | product=424432 | retrieval=0.6974 | rerank=1.0004
3. Cute 3p Short Sock | product=716219 | retrieval=0.6943 | rerank=0.9213
4. Short 5p Socks(1) | product=892079 | retrieval=0.7120 | rerank=0.8866
5. Heavy Check 2p Socks | product=761202 | retrieval=0.6933 | rerank=0.8155
Colorful socks
1. 1pk fun/expressive | product=706268 | retrieval=0.7237 | rerank=4.5088
2. 1pk Fun | product=783707 | retrieval=0.7289 | rerank=4.2893
3. Fun Chenille 2p Lounge

## Reranking Stage — Conclusion

The RAG retrieval pipeline now contains a two-stage ranking process:

**Stage 1 — Candidate Retrieval**

Qdrant performs semantic vector search combined with structured metadata
filtering and returns a larger candidate set.

**Stage 2 — Reranking**

The Cross-Encoder evaluates the relevance between the user query and
each candidate product, then reorders the candidates.

The resulting top-ranked products are now ready to be used by the next
RAG stage.

Current pipeline status:

- Data Ingestion: ✅
- Embedding & Indexing: ✅
- Retrieval: ✅
- Reranking: ✅
- Augmentation: ❌
- Generation: ❌

The next stage is **Augmentation**, where the retrieved products will
be transformed into a structured context that can be provided to an
LLM together with the original user query.

## 12. Augmentation & Generation

### Not Implemented Yet

This notebook stops at structured retrieval results (ranked, filtered products with resolved variants). The remaining RAG stages — turning those results into a natural-language answer — are not yet implemented.

Intended flow:

```
Retrieved Products
        ↓
  Context Assembly
        ↓
 Query + Retrieved Context
        ↓
        Prompt
        ↓
         LLM
        ↓
    Final Answer
```

# 5. Augmentation

After retrieval and reranking, we have a small set of highly relevant
products.

The Augmentation stage prepares these retrieved products as context for
the language model.

The purpose of this stage is to provide the LLM with factual information
retrieved from the product database instead of relying only on its
parametric knowledge.

The flow is:

User Query
→ Retrieval
→ Reranking
→ Retrieved Products
→ Context Assembly
→ Prompt
→ LLM

## What Is Augmentation?

Augmentation means enriching the original user query with relevant
information retrieved from the knowledge base.

At this point, the retriever has already identified the most relevant
products.

We now transform those products into a structured context that can be
included in the LLM prompt.

Therefore:

**Retrieval** finds the information.

**Augmentation** prepares and attaches that information to the query.

**Generation** will use that augmented prompt to produce the final answer.

## Retrieved Products as Context

The reranker currently returns the top-ranked products.

Each product contains structured information such as:

- Product name
- Product type
- Product group
- Gender
- Department
- Section
- Garment group
- Available colors
- Perceived colors
- Matching variants
- Article IDs

We will select the relevant product information and convert it into a
context string for the language model.

In [58]:
def build_context(results):
    context_parts = []

    for i, item in enumerate(results, 1):

        variants_text = ", ".join(
            [
                f"{variant['article_id']} ({variant['colour_group_name']})"
                for variant in item["variants"]
            ]
        )

        product_text = f"""
Product {i}
Product Name: {item["product_name"]}
Product Type: {item["product_type"]}
Product Group: {item["product_group"]}
Gender: {item["gender"]}
Department: {item["department"]}
Section: {item["section"]}
Garment Group: {item["garment_group"]}
Available Colors: {", ".join(item["colors"])}
Perceived Colors: {", ".join(item["perceived_colors"])}
Variants: {variants_text}
""".strip()

        context_parts.append(product_text)

    return "\n\n".join(context_parts)

## Building the Retrieved Context

The `build_context()` function converts the ranked product results into
a single structured context.

Each retrieved product is represented separately so that the language
model can distinguish between different products.

The context contains both product-level information and the exact
variants identified during variant resolution.

This gives the generation stage access to:

- Which product was retrieved
- Why it is relevant
- Which colors are available
- Which exact variants correspond to those colors

## Running the Retrieval and Reranking Pipeline

Before building the context, we run the complete retrieval path:

1. The user provides a query.
2. Qdrant retrieves candidate products.
3. Metadata filters restrict the candidates.
4. Variants are resolved when a color is requested.
5. The Cross-Encoder reranks the candidates.
6. The top-ranked products are passed to the augmentation stage.

In [59]:
query = "short socks for sneakers"

candidates = retrieve_products(
    query=query,
    gender="Men",
    color="Black",
    top_k=15
)

reranked = rerank_products(
    query=query,
    results=candidates,
    top_k=5
)

## Creating the Augmented Context

The reranked products are now converted into a context that can be
provided to the language model.

This is the boundary between the **Retrieval/Reranking** stages and the
**Generation** stage.

The system now has two important inputs:

- The original user query
- The retrieved product context

In [60]:
context = build_context(reranked)

print(context)

Product 1
Product Name: 7pk basic R sneaker socks
Product Type: Socks
Product Group: Socks & Tights
Gender: Men
Department: Socks Bin
Section: Men Underwear
Garment Group: Socks and Tights
Available Colors: Black, Dark Blue, Light Turquoise, White
Perceived Colors: Black, Blue, Turquoise, White
Variants: 575347003 (Black)

Product 2
Product Name: Raz shaftless sock 5-pack.
Product Type: Socks
Product Group: Socks & Tights
Gender: Men
Department: Men Sport Acc
Section: Men H&M Sport
Garment Group: Accessories
Available Colors: Black, Grey, White
Perceived Colors: Black, Grey, White
Variants: 892193001 (Black)

Product 3
Product Name: Lawrence Shaftless Sock 5-p
Product Type: Socks
Product Group: Socks & Tights
Gender: Men
Department: Men Sport Acc
Section: Men H&M Sport
Garment Group: Accessories
Available Colors: Black, Dark Grey, Grey, White
Perceived Colors: Black, Grey, White
Variants: 861850002 (Black), 861850006 (Black)

Product 4
Product Name: Lawrence Reflective Low Sock 3
Produ

## Prompt Construction

The final augmented prompt combines three components:

1. **System instructions**
   - Define how the language model should behave.

2. **Retrieved context**
   - Contains the relevant products returned by the retrieval pipeline.

3. **User query**
   - Represents the original user request.

The resulting prompt will be passed to the language model in the
Generation stage.

Structure:

System Instructions
+
Retrieved Context
+
User Query
→
Augmented Prompt

In [61]:
def build_prompt(query, context):
    prompt = f"""
You are a factual e-commerce product assistant.

Your task is to answer the user's query using ONLY the product information
provided in the context below.

STRICT RULES:
1. Do not invent or assume product attributes.
2. Do not claim that a product is comfortable, durable, high quality,
   suitable, reflective, soft, or similar unless the context explicitly
   says so.
3. Do not use outside knowledge.
4. If the requested information is not available in the context, say so.
5. Recommend only products that match the user's explicit requirements.
6. Keep the answer concise.
7. Mention the product name and relevant available colors when useful.

Retrieved Product Context:
{context}

User Query:
{query}

Answer:
""".strip()

    return prompt

The `build_prompt()` function creates the augmented prompt by combining
the retrieved product context with the original user query.

The instruction to use only the retrieved context helps keep product
answers grounded in the information retrieved from the catalog.

At this point, we have completed the Augmentation stage.

The language model has not generated an answer yet.

In [62]:
prompt = build_prompt(
    query=query,
    context=context
)

print(prompt)

You are a factual e-commerce product assistant.

Your task is to answer the user's query using ONLY the product information
provided in the context below.

STRICT RULES:
1. Do not invent or assume product attributes.
2. Do not claim that a product is comfortable, durable, high quality,
   suitable, reflective, soft, or similar unless the context explicitly
   says so.
3. Do not use outside knowledge.
4. If the requested information is not available in the context, say so.
5. Recommend only products that match the user's explicit requirements.
6. Keep the answer concise.
7. Mention the product name and relevant available colors when useful.

Retrieved Product Context:
Product 1
Product Name: 7pk basic R sneaker socks
Product Type: Socks
Product Group: Socks & Tights
Gender: Men
Department: Socks Bin
Section: Men Underwear
Garment Group: Socks and Tights
Available Colors: Black, Dark Blue, Light Turquoise, White
Perceived Colors: Black, Blue, Turquoise, White
Variants: 575347003 (Black)


## Augmentation Stage — Completed

The original user query has now been augmented with information retrieved
from the product knowledge base.

The complete flow is:

User Query
↓
Semantic Retrieval
↓
Metadata Filtering
↓
Variant Resolution
↓
Cross-Encoder Reranking
↓
Top-K Products
↓
Context Assembly
↓
Prompt Construction
↓
Augmented Prompt

The augmented prompt is now ready to be passed to a language model.

The next stage is **Generation**, where the LLM will use the augmented
prompt to produce the final response.

# 6. Generation

The Generation stage is responsible for producing the final response
using the augmented prompt created in the previous stage.

At this point, the RAG pipeline already has:

- Retrieved relevant products.
- Reranked the retrieved products.
- Built a structured product context.
- Constructed an augmented prompt.

The language model will now use this prompt to generate the final answer.

The flow is:

User Query
→ Retrieval
→ Reranking
→ Augmentation
→ Augmented Prompt
→ LLM
→ Final Answer

## Language Model Selection

For the Generation stage, we use the
**Qwen2.5-1.5B-Instruct** language model.

The model is instruction-tuned and lightweight enough to run in a
Kaggle environment.

The model is responsible only for generating the final response.

It does not perform retrieval.

The retrieved product information is provided through the augmented
prompt.

In [63]:
!pip install -q transformers accelerate

In [64]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Load Language Model

The Qwen2.5-1.5B-Instruct model is loaded for causal language
generation.

The model will receive the augmented prompt and generate the final
e-commerce response.

In [65]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Generate Response

The generation function takes the augmented prompt and sends it to the
language model.

The process is:

1. Tokenize the prompt.
2. Pass the tokens to the language model.
3. Generate new tokens.
4. Decode the generated tokens back into text.

The generated response represents the final output of the RAG pipeline.

In [66]:
def generate_response(
    prompt,
    max_new_tokens=200
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

## Generate the Final Answer

The augmented prompt created during the Augmentation stage is now passed
to the language model.

The model uses the retrieved product context to generate the final
answer.

In [67]:
response = generate_response(
    prompt
)

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Based on your request for short socks for sneakers, I recommend the following:

- **Product**: 7pk basic R sneaker socks
- **Color Options**: Black, Dark Blue, Light Turquoise, White

These socks are perfect for men who prefer shorter lengths and look great with sneakers! They come in various colors including black, dark blue, light turquoise, and white, ensuring you have plenty of options to choose from. 

Please note that these socks are part of the "Men Underwear" section within the "Socks Bin." You can find them at any store that carries this type of product. Enjoy your shopping! 🏖️👟 #ShortSocksForSneakers #MenUnderwear #SocksBin #H&M #SneakerFriendly #ComfortableAndDurable #ReflectiveColors #SoftMaterials #SuitableForAllGenders #HighQualityProducts #EcoFriendlyMaterials #HealthyFootCare #ActiveLifestyle #SportAcc #MenSport


## Generation Test

We test the complete retrieval-to-generation process using a product
search query.

The query asks for short black socks suitable for sneakers for men.

The retrieved products are used as context for the language model.

In [68]:
query = "short black socks for sneakers for men"

candidates = retrieve_products(
    query=query,
    gender="Men",
    color="Black",
    top_k=15
)

reranked = rerank_products(
    query=query,
    results=candidates,
    top_k=5
)

context = build_context(
    reranked
)

prompt = build_prompt(
    query=query,
    context=context
)

response = generate_response(
    prompt
)

print(response)

The product matching your request is the 7pk basic R sneaker socks. Available colors include Black, Dark Blue, Light Turquoise, and White. It has variants of 575347003 (Black). 

Note: This answer assumes you want short socks rather than long socks as per the user's query. If you specifically need long socks, please let me know! Also, I've mentioned "sneakers" because it was asked for, but if you're looking for general athletic wear, we could consider other options like Raz shaftless sock 5-pack or Lawrence Reflective Low Sock 3. Let me know how you'd like to proceed! 🚀👟 #MenUnderwear #SportsWear #AthleticGear #ComfortableFootwear #DurableFootwear #ReflectiveFootwear #ShortSocksForSneakers #MenSneakerSocks #H&M #SportAcc #MenSportAcc #MenH&M


## test multiple quaries

In [69]:
generation_tests = [
    {
        "query": "short black socks for sneakers for men",
        "gender": "Men",
        "color": "Black"
    },
    {
        "query": "comfortable white socks for women",
        "gender": "Women",
        "color": "White"
    },
    {
        "query": "colorful fun socks",
        "gender": None,
        "color": None
    },
    {
        "query": "comfortable everyday socks for men",
        "gender": "Men",
        "color": None
    },
    {
        "query": "blue socks",
        "gender": None,
        "color": "Blue"
    }
]

In [70]:
for test in generation_tests:

    print("=" * 80)
    print("QUERY:", test["query"])
    print("=" * 80)

    candidates = retrieve_products(
        query=test["query"],
        gender=test["gender"],
        color=test["color"],
        top_k=15
    )

    reranked = rerank_products(
        query=test["query"],
        results=candidates,
        top_k=5
    )

    context = build_context(
        reranked
    )

    prompt = build_prompt(
        query=test["query"],
        context=context
    )

    response = generate_response(
        prompt
    )

    print(response)
    print()

QUERY: short black socks for sneakers for men
The product matching your request is the 7pk basic R sneaker socks. Available colors include Black, Dark Blue, Light Turquoise, and White. It has variants of 575347003 (Black). 

Note: This answer assumes you want short socks rather than long socks as per the user's query. If you specifically need long socks, please let me know! Also, I've mentioned "sneakers" because it was asked for, but if you're looking for general athletic wear, we could consider other options like Raz shaftless sock 5-pack or Lawrence Reflective Low Sock 3. Let me know how you'd like to proceed! 🚀👟 #MenUnderwear #SportsWear #AthleticGear #ComfortableFootwear #DurableFootwear #ReflectiveFootwear #ShortSocksForSneakers #MenSneakerSocks #H&M #SportAcc #MenSportAcc #MenH&M

QUERY: comfortable white socks for women
The most suitable product for your request is "1p Casual Socks" as it matches all your criteria of being comfortable, white color, and suitable for women. Avail

---

# Part 2 — Embedding Model Experiment

## 2.1 What we are testing, and why score comparison is not enough

The baseline embedding model is `BAAI/bge-small-en-v1.5` (384-dim). We want to
know whether a different embedding model retrieves **better products** for
realistic sock queries.

Cosine similarity scores from two different models live in different spaces and
are **not comparable**. A score of 0.82 from one model means nothing relative to
0.71 from another. So the experiment compares *which products come back*, judged
against the structured metadata we already have.

## 2.2 A subtlety that matters: query prefixes

BGE and E5 models are trained with **asymmetric prompts** — the query is
supposed to be prefixed before encoding:

- BGE: `"Represent this sentence for searching relevant passages: "` on the query only
- E5: `"query: "` on queries, `"passage: "` on documents

The current notebook does **not** apply the BGE query prefix. That is a usage
issue, not a model issue, and it would be unfair to give candidate models their
correct prefix while denying the baseline its own.

So the experiment includes **`bge-small` with its correct prefix as a separate
arm**. This separates two different questions:

- Does the *model* need to change?
- Or does the *existing model* just need to be used correctly?

## 2.3 Candidate models

Chosen for retrieval suitability on short English e-commerce queries, and for
fitting comfortably in Kaggle memory. All are open-source and ungated.

| Arm | Model | Dim | Notes |
|---|---|---|---|
| `baseline` | `BAAI/bge-small-en-v1.5` | 384 | Exactly as currently used — no prefix |
| `bge_small_prefixed` | `BAAI/bge-small-en-v1.5` | 384 | Same model, correct BGE query prefix |
| `bge_base` | `BAAI/bge-base-en-v1.5` | 768 | Same family, larger |
| `e5_base` | `intfloat/e5-base-v2` | 768 | Different family, query:/passage: scheme |
| `gte_base` | `Alibaba-NLP/gte-base-en-v1.5` | 768 | Strong retrieval model; needs `trust_remote_code` |

Larger is not assumed better — that is what the measurement is for. `gte_base`
is included but can be disabled with one flag if `trust_remote_code` causes
trouble in your Kaggle image.


In [71]:
# Embedding experiment configuration.
# Set RUN_GTE = False if trust_remote_code causes problems in your environment.

RUN_GTE = False ## set true

EMBEDDING_ARMS = {
    "baseline": {
        "model_name": "BAAI/bge-small-en-v1.5",
        "query_prefix": "",
        "doc_prefix": "",
        "trust_remote_code": False,
        "label": "bge-small-en-v1.5 (baseline, no prefix)",
    },
    "bge_small_prefixed": {
        "model_name": "BAAI/bge-small-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "doc_prefix": "",
        "trust_remote_code": False,
        "label": "bge-small-en-v1.5 (correct BGE query prefix)",
    },
    "bge_base": {
        "model_name": "BAAI/bge-base-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "doc_prefix": "",
        "trust_remote_code": False,
        "label": "bge-base-en-v1.5",
    },
    "e5_base": {
        "model_name": "intfloat/e5-base-v2",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
        "trust_remote_code": False,
        "label": "e5-base-v2",
    },
}

if RUN_GTE:
    EMBEDDING_ARMS["gte_base"] = {
        "model_name": "Alibaba-NLP/gte-base-en-v1.5",
        "query_prefix": "",
        "doc_prefix": "",
        "trust_remote_code": True,
        "label": "gte-base-en-v1.5",
    }

for key, cfg in EMBEDDING_ARMS.items():
    print(f"{key:20s} -> {cfg['label']}")


baseline             -> bge-small-en-v1.5 (baseline, no prefix)
bge_small_prefixed   -> bge-small-en-v1.5 (correct BGE query prefix)
bge_base             -> bge-base-en-v1.5
e5_base              -> e5-base-v2


## 2.4 Evaluation set

The five existing test cases are kept unchanged. Four additional realistic
e-commerce queries are added to widen coverage — none of the originals are
removed or reworded.

For each case we record what the query *asks for*, using values that exist in
the catalog metadata:

- `gender` / `color` — checkable exactly against the Qdrant payload.
- `intent_keywords` — words that should plausibly appear in a genuinely relevant
  product document (e.g. a sneaker-sock query should surface products whose text
  mentions ankle/trainer/short).

**Honesty note:** `intent_keywords` is a **proxy**, not a human relevance
judgment. A product can be relevant without containing the keyword, and vice
versa. It is reported as a weak signal alongside the exact attribute checks,
which are reliable.


In [72]:
# Evaluation cases.
# The first five are the existing test_cases, unchanged.

evaluation_cases = [
    {
        "name": "Men black sneaker socks",
        "query": "short black socks for sneakers for men",
        "gender": "Men",
        "color": "Black",
        "intent_keywords": ["sneaker", "trainer", "ankle", "short", "low"],
    },
    {
        "name": "Women white socks",
        "query": "comfortable white socks for women",
        "gender": "Women",
        "color": "White",
        "intent_keywords": ["sock"],
    },
    {
        "name": "Colorful socks",
        "query": "colorful fun socks",
        "gender": None,
        "color": None,
        "intent_keywords": ["pattern", "colour", "color", "stripe", "fun", "print"],
    },
    {
        "name": "Men everyday socks",
        "query": "comfortable everyday socks for men",
        "gender": "Men",
        "color": None,
        "intent_keywords": ["sock"],
    },
    {
        "name": "Blue socks",
        "query": "blue socks",
        "gender": None,
        "color": "Blue",
        "intent_keywords": ["sock"],
    },
    # --- additional realistic queries (originals above are untouched) ---
    {
        "name": "Wool winter socks",
        "query": "warm wool socks for winter",
        "gender": None,
        "color": None,
        "intent_keywords": ["wool", "warm", "thick", "knit"],
    },
    {
        "name": "Knee high socks",
        "query": "knee high socks",
        "gender": None,
        "color": None,
        "intent_keywords": ["knee", "long", "high"],
    },
    {
        "name": "Baby socks",
        "query": "socks for babies",
        "gender": None,
        "color": None,
        "intent_keywords": ["baby", "infant", "newborn"],
    },
    {
        "name": "Multipack sports socks",
        "query": "multipack sports socks",
        "gender": None,
        "color": None,
        "intent_keywords": ["pack", "sport", "pk"],
    },
]

print("Evaluation cases:", len(evaluation_cases))
for case in evaluation_cases:
    print(f"  - {case['name']:28s} | {case['query']}")


Evaluation cases: 9
  - Men black sneaker socks      | short black socks for sneakers for men
  - Women white socks            | comfortable white socks for women
  - Colorful socks               | colorful fun socks
  - Men everyday socks           | comfortable everyday socks for men
  - Blue socks                   | blue socks
  - Wool winter socks            | warm wool socks for winter
  - Knee high socks              | knee high socks
  - Baby socks                   | socks for babies
  - Multipack sports socks       | multipack sports socks


## 2.5 Experiment harness

To evaluate a candidate without disturbing the baseline, each arm gets:

- its own embeddings over the **same** `product_documents`
- its own Qdrant collection
- its own `ProductRetriever` (the existing class, reused unchanged)

`retrieve_products_with()` below is the body of the existing
`retrieve_products()` with the retriever passed in as an argument instead of
read from the global. The logic — `top_k * 3` candidate over-fetch, filtering,
variant resolution, skip-on-empty-color — is identical. The original
`retrieve_products()` is left untouched and still works.


In [73]:
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client.models import VectorParams, Distance, PointStruct


def build_arm_index(arm_key, cfg, product_documents, client):
    """Embed the same product documents with one model and index them in
    a dedicated Qdrant collection. Returns the model, dim, and timings."""

    model = SentenceTransformer(
        cfg["model_name"],
        trust_remote_code=cfg["trust_remote_code"],
    )

    documents = [
        cfg["doc_prefix"] + item["document"]
        for item in product_documents
    ]

    t0 = time.time()
    arm_embeddings = model.encode(
        documents,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=32,
    )
    encode_seconds = time.time() - t0

    dim = int(arm_embeddings.shape[1])
    arm_collection = f"hm_socks_{arm_key}"

    if client.collection_exists(arm_collection):
        client.delete_collection(collection_name=arm_collection)

    client.create_collection(
        collection_name=arm_collection,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )

    arm_points = []
    for idx, item in enumerate(product_documents):
        metadata = item["metadata"]
        payload = {
            "product_code": metadata["product_code"],
            "article_ids": metadata["article_ids"],
            "product_name": metadata["product_name"],
            "product_type": metadata["product_type"],
            "product_group": metadata["product_group"],
            "gender": metadata["gender"],
            "department": metadata["department"],
            "section": metadata["section"],
            "garment_group": metadata["garment_group"],
            "colors": metadata["colors"],
            "perceived_colors": metadata["perceived_colors"],
        }
        arm_points.append(
            PointStruct(
                id=idx,
                vector=arm_embeddings[idx].tolist(),
                payload=payload,
            )
        )

    client.upsert(collection_name=arm_collection, points=arm_points)

    return {
        "model": model,
        "collection": arm_collection,
        "dim": dim,
        "n_docs": len(documents),
        "encode_seconds": encode_seconds,
    }


In [74]:
class PrefixedEmbeddingModel:
    """Wraps a SentenceTransformer so that .encode() applies the model's
    official query prefix. ProductRetriever calls .encode(query, ...) for
    queries only, so this makes prefixing transparent to the existing code."""

    def __init__(self, model, query_prefix=""):
        self.model = model
        self.query_prefix = query_prefix

    def encode(self, text, **kwargs):
        if isinstance(text, str):
            text = self.query_prefix + text
        else:
            text = [self.query_prefix + t for t in text]
        return self.model.encode(text, **kwargs)


def retrieve_products_with(
    retriever_obj,
    query,
    gender=None,
    color=None,
    product_type=None,
    department=None,
    section=None,
    top_k=5
):
    """Identical logic to the existing retrieve_products(), with the retriever
    passed in instead of read from the global scope."""

    candidate_k = top_k * 3

    results = retriever_obj.search(
        query=query,
        gender=gender,
        color=color,
        product_type=product_type,
        department=department,
        section=section,
        top_k=candidate_k
    )

    final_results = []

    for result in results:

        payload = result.payload
        product_code = payload["product_code"]

        variants = retriever_obj.resolve_variants(
            product_code=product_code,
            color=color
        )

        if color and variants.empty:
            continue

        final_results.append({
            "score": float(result.score),
            "product_code": product_code,
            "product_name": payload["product_name"],
            "product_type": payload["product_type"],
            "product_group": payload["product_group"],
            "gender": payload["gender"],
            "department": payload["department"],
            "section": payload["section"],
            "garment_group": payload["garment_group"],
            "colors": payload["colors"],
            "perceived_colors": payload["perceived_colors"],
            "variants": variants[
                [
                    "article_id",
                    "colour_group_name",
                    "perceived_colour_master_name"
                ]
            ].to_dict("records")
        })

        if len(final_results) >= top_k:
            break

    return final_results


## 2.6 Retrieval metrics

Two evaluation modes, because they answer different questions:

**Unfiltered mode** — no metadata filters. The requested gender/color appear
only as words in the query, so the model has to understand them. *This is where
embedding quality actually shows up.*

**Filtered mode** — production behaviour, with hard metadata filters applied.
Gender and color precision are trivially 1.0 here because the filter enforces
them, so this mode is used to check ranking and flooding, not attribute
accuracy.

Metrics per case, at K=5:

| Metric | Meaning | Reliability |
|---|---|---|
| `gender_precision` | Share of top-K with the requested gender | Exact |
| `color_precision` | Share of top-K stocking the requested color | Exact |
| `type_precision` | Share of top-K that are sock products | Exact |
| `first_hit_rank` | Rank of the first result satisfying *all* requested attributes (RR = 1/rank) | Exact |
| `keyword_hit` | Share of top-K whose document contains an intent keyword | **Proxy** |
| `distinct_products` | Distinct `product_code` in top-K (flooding check) | Exact |


In [75]:
DOCS_BY_CODE = {
    item["metadata"]["product_code"]: item["document"].lower()
    for item in product_documents
}

SOCK_TYPE_TOKENS = ["sock", "tights", "stocking"]


def is_sock_product(item):
    text = f"{item['product_type']} {item['product_group']}".lower()
    return any(token in text for token in SOCK_TYPE_TOKENS)


def evaluate_retrieval(results, case, k=5):
    """Compute retrieval metrics for one query's results. All metrics are
    derived from catalog metadata - nothing is hand-labelled."""

    top = results[:k]

    if not top:
        return {
            "n_results": 0,
            "gender_precision": np.nan,
            "color_precision": np.nan,
            "type_precision": 0.0,
            "keyword_hit": 0.0,
            "reciprocal_rank": 0.0,
            "distinct_products": 0,
        }

    gender_hits, color_hits = [], []

    for item in top:
        if case["gender"]:
            gender_hits.append(item["gender"] == case["gender"])
        if case["color"]:
            color_hits.append(case["color"] in item["colors"])

    type_hits = [is_sock_product(item) for item in top]

    keyword_hits = []
    for item in top:
        doc = DOCS_BY_CODE.get(item["product_code"], "")
        keyword_hits.append(
            any(kw.lower() in doc for kw in case["intent_keywords"])
        )

    # Rank of first result satisfying every requested attribute
    reciprocal_rank = 0.0
    for rank, item in enumerate(top, 1):
        ok = True
        if case["gender"] and item["gender"] != case["gender"]:
            ok = False
        if case["color"] and case["color"] not in item["colors"]:
            ok = False
        if not is_sock_product(item):
            ok = False
        if ok:
            reciprocal_rank = 1.0 / rank
            break

    return {
        "n_results": len(top),
        "gender_precision": float(np.mean(gender_hits)) if gender_hits else np.nan,
        "color_precision": float(np.mean(color_hits)) if color_hits else np.nan,
        "type_precision": float(np.mean(type_hits)),
        "keyword_hit": float(np.mean(keyword_hits)),
        "reciprocal_rank": reciprocal_rank,
        "distinct_products": len({item["product_code"] for item in top}),
    }


## 2.7 Run the experiment

Each arm is built, evaluated on every case in both modes, then released from GPU
memory before the next arm loads. On a Kaggle T4 this takes a few minutes.


In [76]:
import gc
import pandas as pd

embedding_results = []
embedding_arm_info = {}
per_query_records = []

for arm_key, cfg in EMBEDDING_ARMS.items():

    print("=" * 80)
    print("ARM:", arm_key, "|", cfg["label"])

    try:
        info = build_arm_index(arm_key, cfg, product_documents, qdrant_client)
    except Exception as exc:
        print(f"  SKIPPED - failed to build: {exc}")
        continue

    wrapped_model = PrefixedEmbeddingModel(
        info["model"],
        query_prefix=cfg["query_prefix"],
    )

    arm_retriever = ProductRetriever(
        client=qdrant_client,
        collection_name=info["collection"],
        embedding_model=wrapped_model,
        variant_lookup=variant_lookup,
    )

    embedding_arm_info[arm_key] = {
        **info,
        "retriever": arm_retriever,
        "label": cfg["label"],
    }

    for mode in ["unfiltered", "filtered"]:
        for case in evaluation_cases:

            use_gender = case["gender"] if mode == "filtered" else None
            use_color = case["color"] if mode == "filtered" else None

            t0 = time.time()
            results = retrieve_products_with(
                arm_retriever,
                query=case["query"],
                gender=use_gender,
                color=use_color,
                top_k=5,
            )
            query_seconds = time.time() - t0

            metrics = evaluate_retrieval(results, case, k=5)

            record = {
                "arm": arm_key,
                "label": cfg["label"],
                "mode": mode,
                "case": case["name"],
                "query": case["query"],
                "query_seconds": query_seconds,
                **metrics,
            }
            per_query_records.append(record)
            embedding_results.append(record)

    print(f"  dim={info['dim']} | docs={info['n_docs']} | "
          f"encode={info['encode_seconds']:.1f}s")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

embedding_df = pd.DataFrame(embedding_results)
print("\nCollected", len(embedding_df), "query-level records")


ARM: baseline | bge-small-en-v1.5 (baseline, no prefix)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  dim=384 | docs=474 | encode=0.8s
ARM: bge_small_prefixed | bge-small-en-v1.5 (correct BGE query prefix)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

  dim=768 | docs=474 | encode=2.6s

Collected 72 query-level records


In [77]:
# Per-arm summary. Unfiltered mode is the one that reflects embedding quality.

if len(embedding_df):
    summary_cols = [
        "gender_precision",
        "color_precision",
        "type_precision",
        "keyword_hit",
        "reciprocal_rank",
        "query_seconds",
    ]

    for mode in ["unfiltered", "filtered"]:
        print("=" * 100)
        print("MODE:", mode.upper())
        subset = embedding_df[embedding_df["mode"] == mode]
        table = (
            subset
            .groupby(["arm", "label"])[summary_cols]
            .mean()
            .round(4)
            .sort_values("reciprocal_rank", ascending=False)
        )
        print(table.to_string())
        print()
else:
    print("No embedding results - run the previous cell first.")


MODE: UNFILTERED
                                                                 gender_precision  color_precision  type_precision  keyword_hit  reciprocal_rank  query_seconds
arm                label                                                                                                                                       
baseline           bge-small-en-v1.5 (baseline, no prefix)                 0.7333           0.7333             1.0       0.9778           0.8611         0.0168
e5_base            e5-base-v2                                              0.9333           0.5333             1.0       0.9778           0.8333         0.0185
bge_small_prefixed bge-small-en-v1.5 (correct BGE query prefix)            0.7333           0.7333             1.0       1.0000           0.8148         0.0190
bge_base           bge-base-en-v1.5                                        0.6000           0.6667             1.0       1.0000           0.7593         0.0182

MODE: FILTERED
       

In [78]:
# Per-query detail for the unfiltered mode, so individual wins/losses are visible
# rather than hidden inside an average.

if len(embedding_df):
    detail = embedding_df[embedding_df["mode"] == "unfiltered"].pivot_table(
        index="case",
        columns="arm",
        values="reciprocal_rank",
    ).round(3)

    print("Reciprocal rank of first fully-matching product (unfiltered):")
    print(detail.to_string())
else:
    print("No embedding results yet.")


Reciprocal rank of first fully-matching product (unfiltered):
arm                      baseline  bge_base  bge_small_prefixed  e5_base
case                                                                    
Baby socks                   1.00     1.000               1.000      1.0
Blue socks                   0.25     0.000               0.333      0.0
Colorful socks               1.00     1.000               1.000      1.0
Knee high socks              1.00     1.000               1.000      1.0
Men black sneaker socks      0.50     0.333               0.500      1.0
Men everyday socks           1.00     0.500               0.500      0.5
Multipack sports socks       1.00     1.000               1.000      1.0
Women white socks            1.00     1.000               1.000      1.0
Wool winter socks            1.00     1.000               1.000      1.0


## 2.8 Side-by-side retrieved products

Averages hide what actually changed. This prints the top-5 products each arm
returns for the same query, so the difference can be inspected directly.


In [79]:
def compare_arms_on_query(case, mode="unfiltered", k=5):
    use_gender = case["gender"] if mode == "filtered" else None
    use_color = case["color"] if mode == "filtered" else None

    print("=" * 100)
    print("QUERY:", case["query"], f"| mode={mode}")
    print("EXPECTS: gender=", case["gender"], "| color=", case["color"])
    print("=" * 100)

    for arm_key, info in embedding_arm_info.items():
        print(f"\n--- {arm_key}: {info['label']} ---")
        results = retrieve_products_with(
            info["retriever"],
            query=case["query"],
            gender=use_gender,
            color=use_color,
            top_k=k,
        )
        for i, item in enumerate(results, 1):
            flag = ""
            if case["gender"] and item["gender"] != case["gender"]:
                flag += " [gender X]"
            if case["color"] and case["color"] not in item["colors"]:
                flag += " [color X]"
            print(f"  {i}. {item['product_name']} | {item['gender']} | "
                  f"{item['product_type']}{flag}")


if embedding_arm_info:
    for case in evaluation_cases[:3]:
        compare_arms_on_query(case)
        print()
else:
    print("Run the experiment cell first.")


QUERY: short black socks for sneakers for men | mode=unfiltered
EXPECTS: gender= Men | color= Black

--- baseline: bge-small-en-v1.5 (baseline, no prefix) ---
  1. Everyday Sneaker 5p | Women | Socks [gender X]
  2. 7pk basic R sneaker socks | Men | Socks
  3. 1 pk Sneaker | Men | Socks
  4. 3p Sneaker Socks | Women | Socks [gender X]
  5. Micro 2p minishaftless | Women | Socks [gender X]

--- bge_small_prefixed: bge-small-en-v1.5 (correct BGE query prefix) ---
  1. Everyday Sneaker 5p | Women | Socks [gender X]
  2. 7pk basic R sneaker socks | Men | Socks
  3. 3p Sneaker Socks | Women | Socks [gender X]
  4. Quickdry 5p Shaftless (1) | Women | Socks [gender X]
  5. 1 pk Sneaker | Men | Socks

--- bge_base: bge-base-en-v1.5 ---
  1. 3p Sneaker Socks | Women | Socks [gender X]
  2. Everyday Sneaker 5p | Women | Socks [gender X]
  3. 7pk basic R sneaker socks | Men | Socks
  4. Sigge sneaker sock 5p | Unisex/Unknown | Socks [gender X]
  5. 1 pk Sneaker | Men | Socks

--- e5_base: e5-base

## 2.9 Embedding decision

The winner is chosen by a composite of the **unfiltered** metrics — the mode
that actually reflects embedding quality — weighted as:

- reciprocal rank (does a correct product appear, and how high) — 0.40
- gender precision — 0.20
- color precision — 0.20
- type precision — 0.10
- keyword hit (proxy, deliberately small weight) — 0.10

Weights are stated up front rather than tuned after seeing results. The cell
prints the ranking and selects the top arm; ties and near-ties are flagged so a
larger model is not adopted for a negligible gain.


In [80]:
WEIGHTS = {
    "reciprocal_rank": 0.40,
    "gender_precision": 0.20,
    "color_precision": 0.20,
    "type_precision": 0.10,
    "keyword_hit": 0.10,
}

SELECTED_EMBEDDING_ARM = None

if len(embedding_df):
    unfiltered = embedding_df[embedding_df["mode"] == "unfiltered"]
    agg = unfiltered.groupby(["arm", "label"])[list(WEIGHTS)].mean()

    # NaN means the metric did not apply to a case (e.g. no color requested);
    # treat as neutral by filling with the column mean across arms.
    agg_filled = agg.fillna(agg.mean())

    composite = sum(agg_filled[m] * w for m, w in WEIGHTS.items())
    agg_filled["composite"] = composite
    ranked = agg_filled.sort_values("composite", ascending=False)

    print("COMPOSITE RANKING (unfiltered mode)")
    print("=" * 100)
    print(ranked.round(4).to_string())

    best_arm = ranked.index[0][0]
    best_score = ranked["composite"].iloc[0]

    baseline_score = (
        ranked.xs("baseline", level="arm")["composite"].iloc[0]
        if "baseline" in ranked.index.get_level_values("arm")
        else float("nan")
    )

    print("\n" + "=" * 100)
    print(f"Best arm:      {best_arm}  (composite {best_score:.4f})")
    print(f"Baseline arm:  baseline  (composite {baseline_score:.4f})")

    delta = best_score - baseline_score
    print(f"Improvement over baseline: {delta:+.4f}")

    if best_arm == "baseline":
        print("\nDECISION: keep the baseline embedding model - no candidate beat it.")
        SELECTED_EMBEDDING_ARM = "baseline"
    elif delta < 0.02:
        print("\nDECISION: improvement is within noise (<0.02) on a 9-query set.")
        print("Keeping the baseline; the gain does not justify a larger model.")
        SELECTED_EMBEDDING_ARM = "baseline"
    else:
        print(f"\nDECISION: adopt '{best_arm}'.")
        SELECTED_EMBEDDING_ARM = best_arm

    print("=" * 100)
else:
    print("No results - cannot select a model. Run the experiment first.")


COMPOSITE RANKING (unfiltered mode)
                                                                 reciprocal_rank  gender_precision  color_precision  type_precision  keyword_hit  composite
arm                label                                                                                                                                   
baseline           bge-small-en-v1.5 (baseline, no prefix)                0.8611            0.7333           0.7333             1.0       0.9778     0.8356
e5_base            e5-base-v2                                             0.8333            0.9333           0.5333             1.0       0.9778     0.8244
bge_small_prefixed bge-small-en-v1.5 (correct BGE query prefix)           0.8148            0.7333           0.7333             1.0       1.0000     0.8193
bge_base           bge-base-en-v1.5                                       0.7593            0.6000           0.6667             1.0       1.0000     0.7570

Best arm:      baseline  (c

### Note on the 0.02 threshold

Nine queries is a small evaluation set. A composite difference below 0.02 can be
produced by a single query moving one rank, so the notebook treats that as noise
and keeps the baseline. This is deliberately conservative: adopting a 2×-larger
model on the strength of one query's movement would not be a real improvement.

If you extend `evaluation_cases`, this threshold should be revisited.


---

# Part 3 — Generation Model Experiment

## 3.1 Isolating the generation model

To measure only the generation model, every arm receives **identical input**:
the same queries, the same retrieved and reranked products, and the same context
string. Retrieval is run **once**, frozen, and replayed for every model.

## 3.2 A usage issue, same as the prefix issue

The existing `generate_response()` feeds the raw prompt string straight to the
tokenizer. `Qwen2.5-1.5B-Instruct` is an **instruction-tuned** model that expects
its chat template. Skipping the template degrades instruction-following, which is
exactly what grounding depends on.

As with the embedding prefix, this is handled as a separate arm rather than a
silent fix: the baseline model is evaluated **both** as currently invoked and
with its chat template applied. That separates "the model is too small" from
"the model is being called wrong".

The original `generate_response()` is left untouched.

## 3.3 Candidate models

| Arm | Model | Invocation |
|---|---|---|
| `baseline` | `Qwen/Qwen2.5-1.5B-Instruct` | Raw prompt (as currently used) |
| `baseline_chat` | `Qwen/Qwen2.5-1.5B-Instruct` | Chat template |
| `qwen3b` | `Qwen/Qwen2.5-3B-Instruct` | Chat template |
| `qwen7b_4bit` | `Qwen/Qwen2.5-7B-Instruct` | Chat template, 4-bit quantized |

All are ungated and fit a Kaggle T4 (7B only in 4-bit). Set `RUN_7B = False` if
you hit memory limits or want a faster run.


In [81]:
RUN_7B = True

GENERATION_ARMS = {
    "baseline": {
        "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        "use_chat_template": False,
        "load_in_4bit": False,
        "label": "Qwen2.5-1.5B-Instruct (baseline, raw prompt)",
    },
    "baseline_chat": {
        "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        "use_chat_template": True,
        "load_in_4bit": False,
        "label": "Qwen2.5-1.5B-Instruct (chat template)",
    },
    "qwen3b": {
        "model_name": "Qwen/Qwen2.5-3B-Instruct",
        "use_chat_template": True,
        "load_in_4bit": False,
        "label": "Qwen2.5-3B-Instruct",
    },
}

if RUN_7B:
    GENERATION_ARMS["qwen7b_4bit"] = {
        "model_name": "Qwen/Qwen2.5-7B-Instruct",
        "use_chat_template": True,
        "load_in_4bit": True,
        "label": "Qwen2.5-7B-Instruct (4-bit)",
    }

for key, cfg in GENERATION_ARMS.items():
    print(f"{key:16s} -> {cfg['label']}")


baseline         -> Qwen2.5-1.5B-Instruct (baseline, raw prompt)
baseline_chat    -> Qwen2.5-1.5B-Instruct (chat template)
qwen3b           -> Qwen2.5-3B-Instruct
qwen7b_4bit      -> Qwen2.5-7B-Instruct (4-bit)


## 3.4 Freeze the retrieval inputs

Retrieval runs once here using the **baseline** retrieval stack, and the
resulting contexts are cached. Every generation arm then sees byte-identical
input. The existing `retrieve_products()`, `rerank_products()`,
`build_context()`, and `build_prompt()` are used unchanged.


In [82]:
# Run retrieval + reranking ONCE and freeze it, so the generation comparison
# isolates the generation model.

frozen_generation_inputs = []

for test in generation_tests:

    candidates = retrieve_products(
        query=test["query"],
        gender=test["gender"],
        color=test["color"],
        top_k=15
    )

    reranked = rerank_products(
        query=test["query"],
        results=candidates,
        top_k=5
    )

    context = build_context(reranked)
    prompt = build_prompt(query=test["query"], context=context)

    frozen_generation_inputs.append({
        "query": test["query"],
        "gender": test["gender"],
        "color": test["color"],
        "reranked": reranked,
        "context": context,
        "prompt": prompt,
    })

print("Frozen generation inputs:", len(frozen_generation_inputs))
for item in frozen_generation_inputs:
    print(f"  - {item['query']:45s} | context chars: {len(item['context'])}")


Frozen generation inputs: 5
  - short black socks for sneakers for men        | context chars: 1543
  - comfortable white socks for women             | context chars: 1418
  - colorful fun socks                            | context chars: 4549
  - comfortable everyday socks for men            | context chars: 1664
  - blue socks                                    | context chars: 2246


## 3.5 Generation metrics

Subjective reading of a few outputs is not enough, so these rule-based checks
run over every response. Each one is computed against the frozen context, so
"unsupported" means *literally absent from the context that was provided*.

| Metric | How it is computed | Direction |
|---|---|---|
| `unsupported_claims` | Count of flagged quality adjectives present in the response but absent from the context | lower better |
| `hallucinated_products` | Catalog product names appearing in the response but not in this query's context | lower better |
| `hallucinated_ids` | 6+ digit IDs in the response not present in the context | lower better |
| `context_coverage` | Share of context products actually mentioned | higher better |
| `color_mentioned` | Whether the requested color appears (when one was requested) | higher better |
| `marketing_noise` | Count of emojis + hashtags | lower better |
| `word_count` | Response length | context for conciseness |

**What these do not measure:** fluency, helpfulness, and whether the
recommendation is genuinely *good*. Those need human judgment — Section 3.7
prints full outputs for exactly that reason. No metric here is a substitute for
reading them.


In [83]:
import re

# NOTE: terms must not be substrings of one another, or a single word gets
# counted twice (e.g. "comfort" would also fire inside "comfortable").
UNSUPPORTED_TERMS = [
    "comfortable", "durable", "high quality", "high-quality",
    "soft", "reflective", "suitable", "perfect", "best",
    "breathable", "premium", "cozy", "warm", "stylish", "excellent",
    "long-lasting", "moisture", "cushioned", "supportive",
]

ALL_PRODUCT_NAMES = {
    item["metadata"]["product_name"].lower()
    for item in product_documents
    if len(str(item["metadata"]["product_name"])) >= 4
}

EMOJI_PATTERN = re.compile(
    "[" 
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+",
    flags=re.UNICODE,
)


def evaluate_generation(response, context, case):
    """Rule-based grounding checks. Every check is relative to the context
    that was actually supplied for this query."""

    resp_lower = response.lower()
    ctx_lower = context.lower()

    # 1. Quality adjectives asserted without support in the context
    unsupported = [
        term for term in UNSUPPORTED_TERMS
        if term in resp_lower and term not in ctx_lower
    ]

    # 2. Catalog products named but not retrieved for this query
    hallucinated_products = [
        name for name in ALL_PRODUCT_NAMES
        if name in resp_lower and name not in ctx_lower
    ]

    # 3. Article-id-like numbers not present in the context
    resp_ids = set(re.findall(r"\b\d{6,}\b", response))
    ctx_ids = set(re.findall(r"\b\d{6,}\b", context))
    hallucinated_ids = sorted(resp_ids - ctx_ids)

    # 4. How much of the retrieved context the answer actually used
    ctx_product_names = re.findall(r"Product Name: (.+)", context)
    mentioned = [
        name for name in ctx_product_names
        if name.strip().lower() in resp_lower
    ]
    coverage = len(mentioned) / len(ctx_product_names) if ctx_product_names else 0.0

    # 5. Did it respect the requested color
    if case["color"]:
        color_mentioned = case["color"].lower() in resp_lower
    else:
        color_mentioned = np.nan

    # 6. Marketing noise
    emojis = len(EMOJI_PATTERN.findall(response))
    hashtags = len(re.findall(r"#\w+", response))

    return {
        "unsupported_claims": len(unsupported),
        "unsupported_terms": ", ".join(unsupported[:6]),
        "hallucinated_products": len(hallucinated_products),
        "hallucinated_ids": len(hallucinated_ids),
        "context_coverage": round(coverage, 3),
        "color_mentioned": color_mentioned,
        "marketing_noise": emojis + hashtags,
        "word_count": len(response.split()),
        "empty_response": len(response.strip()) == 0,
    }


In [84]:
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_generation_model(cfg):
    tok = AutoTokenizer.from_pretrained(cfg["model_name"])

    kwargs = {"torch_dtype": "auto", "device_map": "auto"}

    if cfg["load_in_4bit"]:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        kwargs.pop("torch_dtype")

    mdl = AutoModelForCausalLM.from_pretrained(cfg["model_name"], **kwargs)
    return tok, mdl


def generate_with(tok, mdl, prompt, use_chat_template, max_new_tokens=200):
    """Greedy decoding (do_sample=False), matching the existing
    generate_response() settings so decoding is not a confounder."""

    if use_chat_template:
        messages = [{"role": "user", "content": prompt}]
        text = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = prompt

    inputs = tok(text, return_tensors="pt").to(mdl.device)

    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tok.decode(generated_tokens, skip_special_tokens=True).strip()


In [85]:
generation_records = []
generation_outputs = {}

for arm_key, cfg in GENERATION_ARMS.items():

    print("=" * 80)
    print("ARM:", arm_key, "|", cfg["label"])

    try:
        tok, mdl = load_generation_model(cfg)
    except Exception as exc:
        print(f"  SKIPPED - failed to load: {exc}")
        continue

    generation_outputs[arm_key] = []

    for item in frozen_generation_inputs:

        t0 = time.time()
        try:
            response = generate_with(
                tok, mdl, item["prompt"], cfg["use_chat_template"]
            )
        except Exception as exc:
            print(f"  generation failed on '{item['query']}': {exc}")
            response = ""
        gen_seconds = time.time() - t0

        case = {"color": item["color"], "gender": item["gender"]}
        metrics = evaluate_generation(response, item["context"], case)

        generation_records.append({
            "arm": arm_key,
            "label": cfg["label"],
            "query": item["query"],
            "gen_seconds": round(gen_seconds, 2),
            **metrics,
        })

        generation_outputs[arm_key].append({
            "query": item["query"],
            "response": response,
        })

    print(f"  done - {len(frozen_generation_inputs)} queries")

    del mdl, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

generation_df = pd.DataFrame(generation_records)
print("\nCollected", len(generation_df), "generation records")


ARM: baseline | Qwen2.5-1.5B-Instruct (baseline, raw prompt)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  done - 5 queries
ARM: baseline_chat | Qwen2.5-1.5B-Instruct (chat template)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  done - 5 queries
ARM: qwen3b | Qwen2.5-3B-Instruct


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  SKIPPED - failed to load: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

Collected 15 generation records


In [86]:
if len(generation_df):
    gen_cols = [
        "unsupported_claims",
        "hallucinated_products",
        "hallucinated_ids",
        "context_coverage",
        "color_mentioned",
        "marketing_noise",
        "word_count",
        "gen_seconds",
    ]

    gen_summary = (
        generation_df
        .groupby(["arm", "label"])[gen_cols]
        .mean()
        .round(3)
    )

    print("GENERATION METRICS (averaged over the 5 test queries)")
    print("=" * 110)
    print(gen_summary.to_string())
else:
    print("No generation results yet.")


GENERATION METRICS (averaged over the 5 test queries)
                                                            unsupported_claims  hallucinated_products  hallucinated_ids  context_coverage color_mentioned  marketing_noise  word_count  gen_seconds
arm           label                                                                                                                                                                                
baseline      Qwen2.5-1.5B-Instruct (baseline, raw prompt)                 1.2                    0.0               0.0              0.20             1.0              5.0       130.0       11.752
baseline_chat Qwen2.5-1.5B-Instruct (chat template)                        0.0                    0.0               0.0              0.16             1.0              0.0        19.6        3.972
qwen3b        Qwen2.5-3B-Instruct                                          0.2                    0.0               0.2              0.48             1.0         

## 3.6 Generation decision

Grounding is weighted most heavily — for a product assistant, an invented
attribute is worse than a terse answer. The composite is:

- no unsupported claims — 0.35
- no hallucinated products — 0.25
- context coverage — 0.20
- requested color mentioned — 0.10
- no marketing noise — 0.10


In [87]:
SELECTED_GENERATION_ARM = None

if len(generation_df):
    g = generation_df.groupby(["arm", "label"]).agg({
        "unsupported_claims": "mean",
        "hallucinated_products": "mean",
        "hallucinated_ids": "mean",
        "context_coverage": "mean",
        "color_mentioned": "mean",
        "marketing_noise": "mean",
        "empty_response": "mean",
    })

    # Normalize "lower is better" counts into 0-1 goodness scores
    def inverse_score(series):
        if series.max() == 0:
            return pd.Series(1.0, index=series.index)
        return 1.0 - (series / series.max())

    score = (
        0.35 * inverse_score(g["unsupported_claims"])
        + 0.25 * inverse_score(g["hallucinated_products"] + g["hallucinated_ids"])
        + 0.20 * g["context_coverage"].fillna(0)
        + 0.10 * g["color_mentioned"].fillna(0)
        + 0.10 * inverse_score(g["marketing_noise"])
    )

    # An arm that returns empty responses cannot win
    score = score * (1.0 - g["empty_response"])

    g["composite"] = score
    ranked_gen = g.sort_values("composite", ascending=False)

    print("GENERATION COMPOSITE RANKING")
    print("=" * 110)
    print(ranked_gen.round(4).to_string())

    best_gen = ranked_gen.index[0][0]
    best_gen_score = ranked_gen["composite"].iloc[0]
    baseline_gen_score = (
        ranked_gen.xs("baseline", level="arm")["composite"].iloc[0]
        if "baseline" in ranked_gen.index.get_level_values("arm")
        else float("nan")
    )

    print("\n" + "=" * 110)
    print(f"Best arm:     {best_gen} (composite {best_gen_score:.4f})")
    print(f"Baseline arm: baseline (composite {baseline_gen_score:.4f})")
    print(f"Improvement:  {best_gen_score - baseline_gen_score:+.4f}")

    if best_gen == "baseline":
        print("\nDECISION: keep the baseline generation model.")
    else:
        print(f"\nDECISION: adopt '{best_gen}'.")

    SELECTED_GENERATION_ARM = best_gen
    print("=" * 110)
    print("\nNOTE: only 5 queries. Read the full outputs below before trusting this.")
else:
    print("No generation results - cannot select a model.")


GENERATION COMPOSITE RANKING
                                                            unsupported_claims  hallucinated_products  hallucinated_ids  context_coverage color_mentioned  marketing_noise  empty_response  composite
arm           label                                                                                                                                                                                  
baseline_chat Qwen2.5-1.5B-Instruct (chat template)                        0.0                    0.0               0.0              0.16             1.0              0.0             0.0     0.8320
qwen3b        Qwen2.5-3B-Instruct                                          0.2                    0.0               0.2              0.48             1.0              0.0             0.0     0.5877
baseline      Qwen2.5-1.5B-Instruct (baseline, raw prompt)                 1.2                    0.0               0.0              0.20             1.0              5.0         

/tmp/ipykernel_58/2597276286.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  + 0.10 * g["color_mentioned"].fillna(0)


## 3.7 Full outputs — read these

The metrics above catch grounding failures, not bad writing. This prints every
arm's full answer for each query so the numbers can be checked against reality.


In [88]:
for q_idx, item in enumerate(frozen_generation_inputs):
    print("=" * 110)
    print("QUERY:", item["query"])
    print("=" * 110)

    for arm_key in generation_outputs:
        record = generation_outputs[arm_key][q_idx]
        row = generation_df[
            (generation_df["arm"] == arm_key)
            & (generation_df["query"] == item["query"])
        ]

        print(f"\n--- {arm_key} ---")
        if len(row):
            r = row.iloc[0]
            print(f"[unsupported={r['unsupported_claims']} "
                  f"({r['unsupported_terms']}) | "
                  f"halluc_products={r['hallucinated_products']} | "
                  f"coverage={r['context_coverage']} | "
                  f"words={r['word_count']}]")
        print(record["response"])
    print()


QUERY: short black socks for sneakers for men

--- baseline ---
[unsupported=2 (comfortable, durable) | halluc_products=0 | coverage=0.4 | words=109]
The product matching your request is the 7pk basic R sneaker socks. Available colors include Black, Dark Blue, Light Turquoise, and White. It has variants of 575347003 (Black). 

Note: This answer assumes you want short socks rather than long socks as per the user's query. If you specifically need long socks, please let me know! Also, I've mentioned "sneakers" because it was asked for, but if you're looking for general athletic wear, we could consider other options like Raz shaftless sock 5-pack or Lawrence Reflective Low Sock 3. Let me know how you'd like to proceed! 🚀👟 #MenUnderwear #SportsWear #AthleticGear #ComfortableFootwear #DurableFootwear #ReflectiveFootwear #ShortSocksForSneakers #MenSneakerSocks #H&M #SportAcc #MenSportAcc #MenH&M

--- baseline_chat ---
[unsupported=0 () | halluc_products=0 | coverage=0.2 | words=6]
7pk basic R

---

# Part 4 — Final Pipeline

## 4.1 Applying the selected models

The winners from Parts 2 and 3 are bound to the pipeline globals here. Because
the selection is read from `SELECTED_EMBEDDING_ARM` and
`SELECTED_GENERATION_ARM`, this cell adapts to whatever the experiments actually
measured — including keeping the baseline if nothing beat it.

Rebinding the globals means the **existing** `retrieve_products()`,
`rerank_products()`, `build_context()`, and `build_prompt()` keep working
unchanged against the selected models.


In [89]:
# Bind the experimentally selected embedding model into the pipeline globals.

if SELECTED_EMBEDDING_ARM and SELECTED_EMBEDDING_ARM in embedding_arm_info:
    selected_info = embedding_arm_info[SELECTED_EMBEDDING_ARM]

    embedding_model = selected_info["retriever"].embedding_model
    collection_name = selected_info["collection"]
    retriever = selected_info["retriever"]

    print("Embedding model in use:", selected_info["label"])
    print("Collection:", collection_name)
    print("Dimension:", selected_info["dim"])
else:
    print("No selection made - pipeline still uses the original baseline objects.")
    print("Embedding model in use: BAAI/bge-small-en-v1.5 (baseline)")


Embedding model in use: bge-small-en-v1.5 (baseline, no prefix)
Collection: hm_socks_baseline
Dimension: 384


In [90]:
# Load the selected generation model for the final pipeline.

if SELECTED_GENERATION_ARM and SELECTED_GENERATION_ARM in GENERATION_ARMS:
    final_gen_cfg = GENERATION_ARMS[SELECTED_GENERATION_ARM]
    final_tokenizer, final_model = load_generation_model(final_gen_cfg)
    FINAL_USE_CHAT_TEMPLATE = final_gen_cfg["use_chat_template"]
    print("Generation model in use:", final_gen_cfg["label"])
else:
    final_tokenizer, final_model = tokenizer, model
    FINAL_USE_CHAT_TEMPLATE = False
    print("No selection made - using the original baseline generation model.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generation model in use: Qwen2.5-1.5B-Instruct (chat template)


## 4.2 `rag_pipeline()`

One call covering the full flow:

```
User Query
    ↓  Query Embedding
    ↓  Qdrant Retrieval
    ↓  Metadata Filtering
    ↓  Variant Resolution
    ↓  Top-N Candidates
    ↓  Cross-Encoder Reranking
    ↓  Top-K Products
    ↓  Context Assembly
    ↓  Grounded Prompt
    ↓  Selected Generation Model
Final Answer
```

Built entirely from the existing functions — no retrieval, filtering, or
reranking logic is reimplemented.


In [91]:
def rag_pipeline(
    query,
    gender=None,
    color=None,
    product_type=None,
    department=None,
    section=None,
    top_k=5,
    candidate_k=15,
    max_new_tokens=200,
    return_details=True,
):
    """End-to-end RAG pipeline using the experimentally selected models.

    Composed from the existing pipeline functions:
    retrieve_products -> rerank_products -> build_context -> build_prompt -> LLM
    """

    # 1-4. Retrieval, metadata filtering, variant resolution
    candidates = retrieve_products(
        query=query,
        gender=gender,
        color=color,
        product_type=product_type,
        department=department,
        section=section,
        top_k=candidate_k,
    )

    # 5. Cross-encoder reranking
    reranked = rerank_products(
        query=query,
        results=candidates,
        top_k=top_k,
    )

    # 6. Context assembly
    context = build_context(reranked)

    # 7. Grounded prompt
    prompt = build_prompt(query=query, context=context)

    # 8. Generation
    answer = generate_with(
        final_tokenizer,
        final_model,
        prompt,
        FINAL_USE_CHAT_TEMPLATE,
        max_new_tokens=max_new_tokens,
    )

    if not return_details:
        return answer

    return {
        "query": query,
        "n_candidates": len(candidates),
        "products": reranked,
        "context": context,
        "prompt": prompt,
        "answer": answer,
    }


print("rag_pipeline() ready")


rag_pipeline() ready


## 4.3 Final tests

The five original test cases, run end-to-end through the selected pipeline.


In [92]:
for test in generation_tests:

    result = rag_pipeline(
        query=test["query"],
        gender=test["gender"],
        color=test["color"],
        top_k=5,
    )

    print("=" * 100)
    print("QUERY:", result["query"])
    print("=" * 100)

    print("\nRERANKED PRODUCTS:")
    for i, item in enumerate(result["products"], 1):
        print(f"  {i}. {item['product_name']} | code={item['product_code']} | "
              f"{item['gender']} | rerank={item['rerank_score']:.4f}")
        for variant in item["variants"][:3]:
            print(f"       -> {variant['article_id']} | {variant['colour_group_name']}")

    print("\nFINAL ANSWER:")
    print(result["answer"])
    print()


QUERY: short black socks for sneakers for men

RERANKED PRODUCTS:
  1. 7pk basic R sneaker socks | code=575347 | Men | rerank=2.7497
       -> 575347003 | Black
  2. Raz shaftless sock 5-pack. | code=892193 | Men | rerank=2.4973
       -> 892193001 | Black
  3. Lawrence Shaftless Sock 5-p | code=861850 | Men | rerank=2.4403
       -> 861850002 | Black
       -> 861850006 | Black
  4. Lawrence Reflective Low Sock 3 | code=862122 | Men | rerank=2.3122
       -> 862122001 | Black
  5. 1 pk Sneaker | code=720577 | Men | rerank=2.1186
       -> 720577008 | Black

FINAL ANSWER:
7pk basic R sneaker socks (Black)

QUERY: comfortable white socks for women

RERANKED PRODUCTS:
  1. 1p Casual Socks | code=807239 | Women | rerank=3.3232
       -> 807239005 | White
  2. Short 5p Socks(1) | code=892079 | Women | rerank=3.0252
       -> 892079001 | White
  3. 1p Placement Socks | code=836809 | Women | rerank=2.9507
       -> 836809003 | White
       -> 836809004 | White
       -> 836809005 | White
  4

## 4.4 Before vs. after

Baseline stack (`bge-small` raw + `Qwen2.5-1.5B` raw prompt) against the
selected stack, on the same queries with the same metrics. If the selection kept
the baseline, this table will show no difference — which is a valid result, not
a failure.


In [93]:
# Before/after comparison, pulled from the measurements already collected.

print("BEFORE vs AFTER")
print("=" * 100)

if len(embedding_df) and SELECTED_EMBEDDING_ARM:
    unf = embedding_df[embedding_df["mode"] == "unfiltered"]
    metric_cols = list(WEIGHTS)

    before_e = unf[unf["arm"] == "baseline"][metric_cols].mean()
    after_e = unf[unf["arm"] == SELECTED_EMBEDDING_ARM][metric_cols].mean()

    comparison_e = pd.DataFrame({
        "baseline": before_e.round(4),
        "selected": after_e.round(4),
        "delta": (after_e - before_e).round(4),
    })
    print("\nRETRIEVAL (unfiltered mode)")
    print(f"  baseline: bge-small-en-v1.5 (no prefix)")
    print(f"  selected: {embedding_arm_info[SELECTED_EMBEDDING_ARM]['label']}")
    print(comparison_e.to_string())
else:
    print("\nRETRIEVAL: no comparison available.")

if len(generation_df) and SELECTED_GENERATION_ARM:
    gen_metric_cols = [
        "unsupported_claims", "hallucinated_products",
        "context_coverage", "marketing_noise", "word_count",
    ]
    before_g = generation_df[generation_df["arm"] == "baseline"][gen_metric_cols].mean()
    after_g = generation_df[
        generation_df["arm"] == SELECTED_GENERATION_ARM
    ][gen_metric_cols].mean()

    comparison_g = pd.DataFrame({
        "baseline": before_g.round(3),
        "selected": after_g.round(3),
        "delta": (after_g - before_g).round(3),
    })
    print("\nGENERATION")
    print(f"  baseline: Qwen2.5-1.5B-Instruct (raw prompt)")
    print(f"  selected: {GENERATION_ARMS[SELECTED_GENERATION_ARM]['label']}")
    print(comparison_g.to_string())
else:
    print("\nGENERATION: no comparison available.")


BEFORE vs AFTER

RETRIEVAL (unfiltered mode)
  baseline: bge-small-en-v1.5 (no prefix)
  selected: bge-small-en-v1.5 (baseline, no prefix)
                  baseline  selected  delta
reciprocal_rank     0.8611    0.8611    0.0
gender_precision    0.7333    0.7333    0.0
color_precision     0.7333    0.7333    0.0
type_precision      1.0000    1.0000    0.0
keyword_hit         0.9778    0.9778    0.0

GENERATION
  baseline: Qwen2.5-1.5B-Instruct (raw prompt)
  selected: Qwen2.5-1.5B-Instruct (chat template)
                       baseline  selected   delta
unsupported_claims          1.2      0.00   -1.20
hallucinated_products       0.0      0.00    0.00
context_coverage            0.2      0.16   -0.04
marketing_noise             5.0      0.00   -5.00
word_count                130.0     19.60 -110.40


## 4.5 Summary

Generated from the measured results, so it cannot drift from what the
experiments actually found.


In [94]:
print("=" * 100)
print("RAG OPTIMIZATION SUMMARY")
print("=" * 100)

print("\nBaseline Embedding:")
print("  BAAI/bge-small-en-v1.5 (384-dim, no query prefix)")

print("\nSelected Embedding:")
if SELECTED_EMBEDDING_ARM and SELECTED_EMBEDDING_ARM in embedding_arm_info:
    sel = embedding_arm_info[SELECTED_EMBEDDING_ARM]
    print(f"  {sel['label']} ({sel['dim']}-dim)")
else:
    print("  (not selected - experiment not run)")

print("\nBaseline Generation:")
print("  Qwen/Qwen2.5-1.5B-Instruct (raw prompt, no chat template)")

print("\nSelected Generation:")
if SELECTED_GENERATION_ARM and SELECTED_GENERATION_ARM in GENERATION_ARMS:
    print(f"  {GENERATION_ARMS[SELECTED_GENERATION_ARM]['label']}")
else:
    print("  (not selected - experiment not run)")

print("\nRetrieval Improvement:")
if len(embedding_df) and SELECTED_EMBEDDING_ARM:
    unf = embedding_df[embedding_df["mode"] == "unfiltered"]
    b = unf[unf["arm"] == "baseline"]
    s = unf[unf["arm"] == SELECTED_EMBEDDING_ARM]
    for m in ["reciprocal_rank", "gender_precision", "color_precision", "type_precision"]:
        bv, sv = b[m].mean(), s[m].mean()
        print(f"  {m:20s} {bv:.4f} -> {sv:.4f}  ({sv - bv:+.4f})")
    if SELECTED_EMBEDDING_ARM == "baseline":
        print("  No candidate beat the baseline by a meaningful margin.")
else:
    print("  (not measured)")

print("\nGeneration Improvement:")
if len(generation_df) and SELECTED_GENERATION_ARM:
    b = generation_df[generation_df["arm"] == "baseline"]
    s = generation_df[generation_df["arm"] == SELECTED_GENERATION_ARM]
    for m in ["unsupported_claims", "hallucinated_products", "context_coverage", "marketing_noise"]:
        bv, sv = b[m].mean(), s[m].mean()
        print(f"  {m:24s} {bv:.3f} -> {sv:.3f}  ({sv - bv:+.3f})")
    if SELECTED_GENERATION_ARM == "baseline":
        print("  No candidate beat the baseline by a meaningful margin.")
else:
    print("  (not measured)")

print("\nMain Observations:")
print("  - Evaluation set is 9 retrieval queries and 5 generation queries.")
print("    Small; differences of 1-2 positions move the averages noticeably.")
print("  - Attribute checks (gender/color/type) are exact; keyword_hit is a proxy.")
print("  - Grounding metrics catch invented attributes, not poor writing.")
print("    Section 3.7 outputs still need to be read.")
print("  - The architecture is unchanged: product-level documents, hybrid")
print("    filtering, and cross-encoder reranking are all preserved.")
print("=" * 100)


RAG OPTIMIZATION SUMMARY

Baseline Embedding:
  BAAI/bge-small-en-v1.5 (384-dim, no query prefix)

Selected Embedding:
  bge-small-en-v1.5 (baseline, no prefix) (384-dim)

Baseline Generation:
  Qwen/Qwen2.5-1.5B-Instruct (raw prompt, no chat template)

Selected Generation:
  Qwen2.5-1.5B-Instruct (chat template)

Retrieval Improvement:
  reciprocal_rank      0.8611 -> 0.8611  (+0.0000)
  gender_precision     0.7333 -> 0.7333  (+0.0000)
  color_precision      0.7333 -> 0.7333  (+0.0000)
  type_precision       1.0000 -> 1.0000  (+0.0000)
  No candidate beat the baseline by a meaningful margin.

Generation Improvement:
  unsupported_claims       1.200 -> 0.000  (-1.200)
  hallucinated_products    0.000 -> 0.000  (+0.000)
  context_coverage         0.200 -> 0.160  (-0.040)
  marketing_noise          5.000 -> 0.000  (-5.000)

Main Observations:
  - Evaluation set is 9 retrieval queries and 5 generation queries.
    Small; differences of 1-2 positions move the averages noticeably.
  - Attri

## 4.6 What this experiment does and does not establish

**Establishes:**

- Whether a candidate embedding model retrieves products matching the requested
  gender, color, and product type more often than the baseline, with filters off.
- Whether a candidate generation model asserts fewer unsupported attributes and
  names fewer products absent from its context.
- Runtime and dimension costs of each option.

**Does not establish:**

- Real relevance. `intent_keywords` is a keyword proxy. Confirming a ranking is
  genuinely better needs human judgments over a larger query set.
- Statistical significance. Nine and five queries are too few for that claim, so
  the notebook does not make it.
- Production behaviour under traffic, ambiguous queries, or queries with no good
  match.

**Reasonable next steps:** a larger labelled query set, reranker comparison
(`bge-reranker-v2-m3` against the current MiniLM), and exposing `intent_keywords`
coverage per query rather than only its mean.

---

## Pipeline status

| Stage | Status |
|---|---|
| 1. Data Ingestion | ✅ Implemented |
| 2. Embedding & Indexing | ✅ Implemented (+ experimental comparison) |
| 3. Retrieval | ✅ Implemented |
| 4. Reranking | ✅ Implemented (preserved, not modified) |
| 5. Augmentation | ✅ Implemented |
| 6. Generation | ✅ Implemented (+ experimental comparison) |
